# FIT5149 S2 2026 — Assessment 1 · Sanctioned Loan Amount Prediction
### Yarra Credit Group — reproducible single-notebook solution (**v6, HD-ready**)

This notebook runs **top-to-bottom from a fresh kernel** on the supplied files and writes the Kaggle
`submission.csv`. It is organised to mirror the marking rubric, and **every quoted number is produced
by the cell above the sentence that quotes it** (spec §4: *"every number you quote must be reproducible
from your submitted code"*).

**What changed from v1 (in response to a critical review).** v2 corrects several v1 defects and adds one
statistically-validated optimisation:
(i) `property_ref` is **not** a unique key — corrected (Part 1.1);
(ii) the constant-prediction floor is the **mean** (\$74,937), not the median (Part 4);
(iii) missingness is reported as a robust **14%→46%** range, not a fragile "factor of 7", and described as
*consistent with* MAR rather than proven MAR (Part 1.3);
(iv) calibration is standardised to **isotonic `cv=5`** so the validated model *is* the submitted one;
(v) the "irreducible floor" is reframed as a **model-based error decomposition** (Part 5.1);
(vi) a **stage-tuned approval classifier** is adopted — the only change that survived paired,
property-clustered bootstrap validation (**Part 8**);
(vii) added a group-CV leakage check and an OOF fairness/robustness audit (Part 8).

**What changed in v4 (evaluation-protocol repair, Part 9).** A second review argued the next version
should fix *how performance is measured*, not search harder. Acting on it: (a) a single **`CONFIG`**
object drives validation, refit and submission, with a row-by-row re-check and a **SHA-256** hash;
(b) the headline OOF number is corrected from a repeat-averaged (implicit 3-model ensemble) figure to a
**single deployed-model** figure; (c) **nested CV** prices in selection uncertainty and the bootstrap is
relabelled **conditional**; (d) an **SSE decomposition** locates the error; (e) the target-encoding claim
is corrected to a true *representation* comparison. Four further optimisation ideas were tested and
**all rejected**. The v4 *model* equals the v2 model — v4 changes what can be claimed, not what is
predicted.

**What changed in v5 (audit hardening & honest inference, Part 10).** A third review asked for auditable
submission integrity and claims stated on the right basis. Acting on it: (a) `CONFIG` is now **executable
factories** shared by validation *and* refit, with an **assertion** that the deployed model equals
`CONFIG` and that the CSV matches a **frozen SHA-256**; (b) preprocessing ablations are reported as
**paired per-fold deltas** with their own small SD (Part 2.1), not against the absolute fold SD;
(c) the decision-dominance figure is restated on a **squared-error basis (~96–97%)** rather than "98% of
RMSE", and "only stage" → "**dominant priority**"; (d) the four uncertainty quantities are **named and
separated** and the ±16 is explicitly *not* a generalisation interval (Part 10.1); (e) two Priority-2
stretch models (joint 5-state, LightGBM native-categorical) are run through a **pre-declared gate** and
**both rejected**. Fifth independent check ⇒ predictions unchanged; `submission.csv` stays byte-identical.

**What changed in v6 (efficient hyperparameter search, Part 11).** A fourth review asked for a *logically
strong and efficient* CV search of the approval classifier, selected on final dollar RMSE. Part 11 builds
an **efficient shared-ratio harness** (cache each fold's ratio prediction once; re-fit only the
classifier per candidate — ~25× fewer ratio fits, and a clean paired comparison) and runs a purposeful
~25-config search over the three levers the review prioritised: complexity×regularisation, learning
schedule, and calibration. **Every candidate is confirmed worse (or tied) than the v5 incumbent on paired
5×3.** Sixth independent check ⇒ predictions unchanged; `submission.csv` byte-identical; the contribution
is the search *method* that proves the incumbent optimal, not a lower RMSE.

| Part | Rubric component | What it establishes |
|---|---|---|
| 1 | Data quality & EDA (20) | the condition of the data, with self-generated evidence |
| 2 | Preprocessing & FE (15) | target-free feature construction, each decision measured |
| 3 | Validation strategy (part of 20) | leakage-free `RepeatedStratifiedKFold`, justified by evidence |
| 4–5 | Modelling & selection (part of 20) | three model *types* compared; why the two-part model wins |
| 6 | Results & interpretation (15) | which variables matter, by four converging methods |
| 7 | Final model + Kaggle (30) | **tuned** calibrated two-part model refit on all data → `submission.csv` |
| 8 | Optimisation & confirmatory validation | tuning, paired bootstrap, calibration lock, leakage & fairness audit |

**Design philosophy.** The target is not a plain continuous number: 26.8% of applications are declined
(exactly \$0) and, once approved, the amount is a near-deterministic *fraction* of the request. The
whole solution is built around that structure rather than fighting it.

In [ ]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, time
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats

from sklearn.model_selection import (KFold, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error, roc_auc_score

RNG = 42; np.random.seed(RNG)
BLUE, ORANGE, GREEN, RED, GREY = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#666666"
plt.rcParams.update({"figure.dpi":110, "font.size":10.5, "axes.titleweight":"bold",
    "axes.spines.top":False, "axes.spines.right":False, "axes.grid":True,
    "grid.color":"#ECECEC", "axes.axisbelow":True})
AUD = FuncFormatter(lambda x,_: f"${x*1e-3:,.0f}k")

# relative paths (spec §5). Place the CSVs beside this notebook.
train = pd.read_csv("training_set.csv")
test  = pd.read_csv("kaggle_test_X.csv")
TARGET = "sanctioned_amount_aud"
y   = train[TARGET].to_numpy(float)
req = train["requested_amount_aud"].to_numpy(float)
print("train", train.shape, "| test", test.shape,
      "| target std (mean-prediction RMSE floor): ${:,.0f}".format(y.std()))

---
# Part 1 — What condition is the data in?

The brief is explicit that marks here are for *what we found and how we established it*, not for
reciting a checklist. Each subsection below states a finding, shows the code that produced the number,
and records the decision it drives.

## 1.1 Integrity: identifiers, duplicates, and hidden sentinels
Before trusting any column we check for duplicate rows, confirm the identifiers are keys, and scan the
numeric columns for **sentinel values** — magic numbers standing in for "missing" that would poison a
model if read as real quantities.

In [ ]:
print("exact duplicate rows:", train.duplicated().sum())
print("application_id unique:", train["application_id"].is_unique,
      "| property_ref unique:", train["property_ref"].is_unique,
      "| property_ref nunique:", train["property_ref"].nunique(), "over", len(train), "rows")
vc = train["property_ref"].value_counts()
print(f"property_ref: {int((vc>1).sum())} refs repeat (max {int(vc.max())}x); "
      f"appears in both train & test: {len(set(train['property_ref']) & set(test['property_ref']))}")
print("\n-999 sentinel scan (numeric columns):")
for c in train.select_dtypes("number").columns:
    n = int((train[c] == -999).sum())
    if n: print(f"  {c:<26s} {n:>4d} rows == -999 ({100*n/len(train):.1f}%)")
neg = [c for c in ["annual_income_aud","requested_amount_aud","property_value_aud",
                   "existing_repayments_aud"] if (train[c] < 0).any()]
print("\nnumeric columns with impossible negatives (excl. -999):",
      [c for c in neg if not (train[c].dropna().eq(-999).any())] or "none beyond the sentinel")

**Found.** No duplicate rows. `application_id` is a unique key, but **`property_ref` is not** — it
has only 999 distinct values across 19,322 rows (each property recurs up to 34 times, and all 999 appear
in both train and test). It is a repeated *entity* identifier, not a row key. We still **drop it as a
predictor** (Part 2), and Part 8.4 confirms with a `StratifiedGroupKFold` check that this repetition
creates **no material leakage** (its split-half approval-rate correlation is only ≈0.04 — a near-random
ID). Three columns carry a `-999` sentinel — `existing_repayments_aud` (0.5%), `has_co_applicant` (0.6%)
and `property_value_aud` (1.0%). **Decision:** convert `-999` to `NaN` so the imputer and the missingness
logic treat these as what they are — missing — rather than as a repayment of *negative nine hundred
ninety-nine dollars*. Part 2.1 measures the effect.

## 1.2 The target is a hurdle, and the positive part is a discrete haircut
This single subsection determines the architecture of the whole solution, so we establish it carefully.

In [ ]:
zero = (y == 0)
print(f"declined (exactly $0): {zero.sum():,} rows = {100*zero.mean():.1f}%")
print(f"approved amount: min ${y[~zero].min():,.0f}  max ${y[~zero].max():,.0f}  (std of full target ${y.std():,.0f})")

ratio = np.where(~zero, y/np.maximum(req,1), np.nan)          # sanctioned / requested, approved rows
grid  = pd.Series(ratio[~zero]).round(2).value_counts(normalize=True).sort_index()
print("\nsanctioned/requested ratio among approved (top values):")
print(grid.head(6).to_string(float_format=lambda v: f'{v:.3f}'))
print("share landing on the {0.65,0.70,0.75,0.80} grid: "
      f"{pd.Series(ratio[~zero]).round(2).isin([.65,.70,.75,.80]).mean():.3f}")

cr = train['credit_rating']
hi = (cr >= 750) & (~zero)
lo = (cr <  750) & (~zero)
print(f"\ncredit_rating gate:  approved & credit>=750 -> mean ratio {ratio[hi.values].mean():.3f}"
      f"   |   approved & credit<750 -> mean ratio {np.nanmean(ratio[lo.values]):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(y, bins=60, color=BLUE); ax[0].axvline(0, color=RED, lw=2)
ax[0].xaxis.set_major_formatter(AUD); ax[0].set_title("Target: a point mass at $0 + a heavy positive tail")
ax[0].set_xlabel("sanctioned_amount_aud"); ax[0].set_ylabel("applications")
gg = pd.Series(ratio[~zero]).round(2); gg = gg[gg.between(0.6,0.85)]
ax[1].hist(gg, bins=25, color=GREEN)
ax[1].set_title("Approved: sanctioned / requested sits on a 4-value grid")
ax[1].set_xlabel("sanctioned / requested"); ax[1].set_ylabel("approved applications")
plt.tight_layout(); plt.show()

**Found.** 26.8% of rows are declined (exactly \$0); the smallest *approved* amount is \$6,236, so
there is a genuine gap between "declined" and "smallest approved" — the zeros are a distinct decision,
not a continuum. Among approved rows, **99.1%** of the sanctioned/requested ratio lands on the discrete
grid **{0.65, 0.70, 0.75, 0.80}**, and the ratio steps up at **credit_rating = 750** (0.68 below → 0.75
above).

**This dictates the model.** The data-generating process is *approve/decline* × *(a credit-gated
haircut) × requested_amount*. A single smooth regressor cannot emit a point mass at \$0 and a step
function simultaneously; a **two-part (hurdle) model** can, and modelling the **bounded ratio** rather
than the dollar amount removes the enormous request-size scale from the regression target. Parts 4–5
show this prediction is borne out.

## 1.3 Missingness: how much, and — more importantly — *why*
The brief's full-marks example is precisely a missingness-mechanism argument, so we quantify both the
rate and its **structure**.

In [ ]:
miss = train.isna().mean().mul(100).round(1)
print("columns with missing values (%):")
print(miss[miss>0].sort_values(ascending=False).to_string())

print("\ncredit_rating missing rate BY occupation_class (with group size n):")
by = (train.assign(m=train['credit_rating'].isna())
           .groupby('occupation_class')['m'].agg(pct=lambda s: round(100*s.mean(),1), n='size')
           .sort_values('pct', ascending=False))
print(by.to_string())
big = by[by['n'] >= 30]                              # ignore n<30 groups (student n=1, etc.)
from scipy.stats import chi2_contingency
chi2, p_mar, _, _ = chi2_contingency(pd.crosstab(train['occupation_class'], train['credit_rating'].isna()))
print(f"\n-> among substantial groups (n>=30): {big['pct'].min():.0f}% -> {big['pct'].max():.0f}% "
      f"(~{big['pct'].max()/big['pct'].min():.1f}x). chi2(occupation, missing) p={p_mar:.0e} "
      "-> structured missingness dependent on OBSERVED occupation, consistent with MAR.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
big['pct'].sort_values().plot.barh(ax=ax, color=BLUE)
ax.set_title("credit_rating missingness by occupation (groups n>=30)")
ax.set_xlabel("% of rows with credit_rating missing"); plt.tight_layout(); plt.show()

**Found.** Eleven columns carry missing values, up to 32% (`property_age_years`). Crucially,
`credit_rating` is missing for **20.5%** of rows, and among the substantial occupation groups the rate
runs from **≈14% for salaried applicants (n≈10,900) to ≈46% for retirees (n≈1,800)** — about a
**4×** spread (a lone student row shows 100%, but n=1 is not evidence). A χ² test rejects independence of
missingness and occupation at **p < 1e-260**. This is *structured missingness that depends on an observed
variable* — **consistent with Missing-At-Random**, though we cannot rule out an unobserved (MNAR)
component from the data alone.

**Decision.** A single global rule (e.g. mean imputation) is indefensible: it would invent a bureau
score for a cohort that structurally has none. We instead (i) impute within the model pipeline (median,
fit per fold), and (ii) — because the *fact of missingness* is tied to who the applicant is — add
explicit **missing-indicator** features. Section 1.4 shows why those indicators carry real signal, and
Part 8 confirms median+indicator beats letting HGB take raw NaNs.

## 1.4 Missingness is *informative*: it predicts the decision
If whether a field is missing correlates with the outcome, then the missingness itself is a feature.

In [ ]:
for c in ["dependants_count", "credit_rating", "monthly_income_aud"]:
    m = train[c].isna()
    print(f"{c:<20s} missing -> decline {100*(y[m.values]==0).mean():4.1f}%   "
          f"present -> decline {100*(y[~m.values]==0).mean():4.1f}%   (n_missing={m.sum():,})")

**Found.** When `dependants_count` is missing the decline rate is **12.8%**, versus **28.0%** when
present — missingness more than *halves* the decline rate. The direction is consistent across the
structurally-missing fields. Operationally this is intuitive: a fast-tracked, low-risk application
simply has fewer fields completed. **Decision confirmed:** keep missing-indicators for
`dependants_count`, `credit_rating`, `monthly_income_aud`, `employment_sector`; Part 6 shows they rank
among the strongest approval predictors, and Part 2.1 shows removing them costs CV RMSE.

## 1.5 Redundancy and corruption: two columns that must not be modelled naively

In [ ]:
both = train['annual_income_aud'].notna() & train['monthly_income_aud'].notna()
r = np.corrcoef(train.loc[both,'annual_income_aud'], 12*train.loc[both,'monthly_income_aud'])[0,1]
print(f"corr(annual_income, 12 x monthly_income) = {r:.5f}  (n={both.sum():,}) "
      "-> the two income fields are the same variable recorded twice")

ct = pd.crosstab(train['occupation_class'], train['income_consistency'])
determined = (ct.gt(0).sum(axis=1) == 1).mean()
print(f"\nincome_consistency is a deterministic function of occupation_class "
      f"(each occupation maps to a single value in {determined*100:.0f}% of occupations) -> redundant")

**Found.** `annual_income_aud` and `12 × monthly_income_aud` correlate at **0.99996** — they are one
variable stored twice, not two signals. `income_consistency` is *perfectly determined* by
`occupation_class` (every occupation maps to a single consistency label). **Decisions:** consolidate the
two income fields into one `income_filled` (using the 12× identity to repair gaps, Part 2), and **drop**
`income_consistency` as fully redundant. We also drop the free identifiers (`application_id`,
`property_ref`) and the protected attribute `applicant_gender` (Part 7.3, fairness).

## 1.6 Implications carried into modelling
Pulling the findings together: **(a)** the target is a hurdle with a credit-gated discrete haircut →
build a two-part model on the *ratio*; **(b)** missingness is structural and informative → impute in-fold
and add indicators, never a global mean; **(c)** `branch_id` (200 levels) and `employment_sector` (18)
are high-cardinality → they need out-of-fold target encoding, which is a leakage risk handled in Part 3;
**(d)** the two income fields and `income_consistency` are redundant → consolidate/drop. Every one of
these is measured, not asserted.

---
# Part 2 — Preprocessing & feature engineering

We separate feature work into two layers, because they have different leakage properties:

* **`build_features` — target-free, per-row construction.** Sentinel repair, income consolidation,
  cyclical dates, missing-indicators. None of these look at the target or at other rows, so they are
  safe to compute once, before splitting.
* **`make_prep` — fitted transforms, learned inside each fold** (Part 3): median imputation, one-hot
  encoding, and *out-of-fold target encoding* of the high-cardinality columns.

In [ ]:
def build_features(df):
    '''Deterministic, target-free feature construction (safe before splitting).'''
    X = df.copy()
    # 1.1 sentinel repair: -999 is 'missing', not a quantity
    for c in ["existing_repayments_aud", "property_value_aud", "has_co_applicant"]:
        X[c] = X[c].replace(-999, np.nan)
    # 1.5 income consolidation via the 12x identity, then a last-resort corrupted-copy fallback
    X["income_filled"] = (X["annual_income_aud"]
                          .fillna(12 * X["monthly_income_aud"])
                          .fillna(X["property_age_years"]))
    # date -> cyclical month + year (day/month order handled explicitly)
    dt = pd.to_datetime(X["application_date"], format="%d/%m/%Y")
    X["app_month_sin"] = np.sin(2*np.pi*dt.dt.month/12)
    X["app_month_cos"] = np.cos(2*np.pi*dt.dt.month/12)
    X["app_year"] = dt.dt.year
    # 1.4 informative-missingness indicators
    for c in ["dependants_count", "credit_rating", "monthly_income_aud", "employment_sector"]:
        X[c + "_missing"] = X[c].isna().astype(int)
    drop = ["application_id", "property_ref", "applicant_gender", "property_age_years",
            "application_date", "annual_income_aud", "monthly_income_aud",
            "income_consistency", TARGET]
    return X.drop(columns=[c for c in drop if c in X])

NUM = ["applicant_age","requested_amount_aud","existing_repayments_aud","dependants_count",
       "credit_rating","prior_default_count","property_value_aud","enquiry_count_12m","income_filled",
       "app_month_sin","app_month_cos","app_year","has_co_applicant","risk_review_flag",
       "dependants_count_missing","credit_rating_missing","monthly_income_aud_missing","employment_sector_missing"]
LOW  = ["occupation_class","residence_area","card_status","property_category","property_area",
        "marketing_channel","expense_flag_a","expense_flag_b"]
HIGH = ["branch_id","employment_sector"]

def to_object(X):                       # pandas 3.0: TargetEncoder needs object, not the string dtype
    X = X.copy()
    for c in LOW + HIGH: X[c] = X[c].astype(object)
    return X

def make_prep(scale=False):
    steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
    return ColumnTransformer([
        ("num",  Pipeline(steps), NUM),
        ("low",  Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                           ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), LOW),
        ("high", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")),
                           ("te",  TargetEncoder(random_state=RNG))]), HIGH)])

# ---------------------------------------------------------------------------
# CONFIG (v5) — one source of truth, expressed as EXECUTABLE FACTORIES.
# The same three factories build every classifier/regressor/calibrator used in
# BOTH validation (Parts 4,8,9,10) and the final refit (Part 7). Nothing downstream
# re-specifies a hyperparameter, so the configuration that is validated is provably
# the configuration that is submitted.
# ---------------------------------------------------------------------------
CONFIG = dict(
    seed        = RNG,
    classifier  = dict(random_state=RNG, learning_rate=0.02, max_iter=700, max_leaf_nodes=15,
                       min_samples_leaf=100, l2_regularization=1.0),   # selected in Part 8.1
    ratio_model = dict(random_state=RNG),                              # defaults (Part 8.3)
    calibration = dict(method="isotonic", cv=5),                       # locked in Part 8.3 / 9.4
    prediction  = "clip(P(approve) * E[ratio] * requested, 0, requested)",
    cv_scheme   = "RepeatedStratifiedKFold(5,3) on is_zero x credit-quintile",
)
def cfg_classifier():   return HistGradientBoostingClassifier(**CONFIG["classifier"])
def cfg_ratio():        return HistGradientBoostingRegressor(**CONFIG["ratio_model"])
def cfg_calibrated():   # calibrated approval pipeline, exactly as deployed
    return CalibratedClassifierCV(Pipeline([("p", make_prep()), ("m", cfg_classifier())]),
                                  method=CONFIG["calibration"]["method"], cv=CONFIG["calibration"]["cv"])
TUNED_CLF = CONFIG["classifier"]                                       # alias used throughout validation
CAL_CV    = CONFIG["calibration"]["cv"]

Xfe = to_object(build_features(train))
print(f"{Xfe.shape[1]} engineered columns -> "
      f"{make_prep().fit_transform(Xfe, y).shape[1]} model features after encoding")
print("CONFIG (single source of truth):")
for k,v in CONFIG.items(): print(f"  {k:<12s}: {v}")

**Why each constructed feature, in lending terms.**

* **`income_filled`** — a lender serviceability model needs *one* income figure; the annual and monthly
  fields are the same number from two systems (1.5), so we reconcile them with the 12× identity and only
  fall back further when both are absent.
* **missing-indicators** — in this data a blank field is a signal about the *application pathway*
  (fast-tracked, low-risk applications are less completely filled, 1.4), so the indicator is a genuine
  risk feature, not a technical artefact.
* **cyclical month (`sin`/`cos`)** — application timing is periodic; encoding month as two continuous
  components lets a model see December and January as adjacent rather than 12 units apart.
* **out-of-fold target encoding of `branch_id`/`employment_sector`** — branch and sector plausibly carry
  approval-propensity information (local policy, sector risk), but 200 one-hot columns are impractical
  and leak; target encoding compresses each to its (out-of-fold) approval signal.

## 2.1 Did each preprocessing decision actually help? (evidence, not faith)
We start from the full feature set and **remove one decision at a time**, scoring the two-part model by
leakage-free CV (single stratified 5-fold here — a fast screening loop; the final comparison in Part 4
uses the full repeated design). A positive Δ means removing the step *hurts*, i.e. the step earns its
place.

In [ ]:
def _strat_key(y_, cr_, n=5):
    y_ = pd.Series(y_).reset_index(drop=True); cr_ = pd.Series(np.asarray(cr_, float)).reset_index(drop=True)
    isz = (y_ == 0).astype(int)
    cb  = pd.qcut(cr_, q=n, labels=False, duplicates="drop").fillna(-1).astype(int)
    return (isz.astype(str) + "_" + cb.astype(str)).to_numpy()
SK   = _strat_key(y, train["credit_rating"])
skf5 = list(StratifiedKFold(5, shuffle=True, random_state=RNG).split(Xfe, SK))   # fast screening loop

def _prep_for(X, high="te"):
    '''ColumnTransformer that adapts to the variant's columns and the high-card representation:
       high="te" (OOF target encoding), "ohe" (one-hot, rare-collapsed), or "drop" (omit branch/sector).'''
    num = [c for c in NUM if c in X.columns]
    blocks = [("num", SimpleImputer(strategy="median"), num),
              ("low", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="NA")),
                                ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), LOW)]
    if high == "te":
        blocks.append(("high", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="NA")),
                                         ("t", TargetEncoder(random_state=RNG))]), HIGH))
    elif high == "ohe":
        blocks.append(("high", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="NA")),
                                         ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=20))]), HIGH))
    # high == "drop": branch/sector are simply not referenced
    return ColumnTransformer(blocks)

def _rh_perfold(Xo, splits, high="te"):
    '''Per-fold two-part RMSE array (needed for PAIRED deltas on identical folds).'''
    sc = []
    for tri, vai in splits:
        Xtr, Xva, ytr = Xo.iloc[tri], Xo.iloc[vai], y[tri]; pos = ytr > 0
        clf = Pipeline([("p", _prep_for(Xo, high)), ("m", HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr, pos.astype(int))
        p   = clf.predict_proba(Xva)[:, 1]
        reg = Pipeline([("p", _prep_for(Xo, high)), ("m", HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos], ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai], p * reg.predict(Xva) * req[vai]))
    return np.array(sc)

def build_variant(drop_decision=None):
    X = train.copy()
    if drop_decision != "sentinel":
        for c in ["existing_repayments_aud","property_value_aud","has_co_applicant"]:
            X[c] = X[c].replace(-999, np.nan)
    X["income_filled"] = (X["annual_income_aud"] if drop_decision=="income"
                          else X["annual_income_aud"].fillna(12*X["monthly_income_aud"]).fillna(X["property_age_years"]))
    dt = pd.to_datetime(X["application_date"], format="%d/%m/%Y")
    if drop_decision != "date":
        X["app_month_sin"]=np.sin(2*np.pi*dt.dt.month/12); X["app_month_cos"]=np.cos(2*np.pi*dt.dt.month/12); X["app_year"]=dt.dt.year
    if drop_decision != "indicators":
        for c in ["dependants_count","credit_rating","monthly_income_aud","employment_sector"]:
            X[c+"_missing"] = X[c].isna().astype(int)
    return to_object(X.drop(columns=[c for c in ["application_id","property_ref","applicant_gender",
        "property_age_years","application_date","annual_income_aud","monthly_income_aud","income_consistency",TARGET] if c in X]))

base = _rh_perfold(Xfe, skf5)   # per-fold RMSE on the 5 fixed folds
print(f"FULL feature set  CV RMSE ${base.mean():,.0f}\n")
print("Each row removes ONE decision and reports the PAIRED per-fold delta (same folds), so the")
print("uncertainty shown is SD(paired diff) -- NOT the ~$1.2k absolute fold SD, which would hide it.\n")
def paired_ab(lab, arr):
    d = arr - base                      # +ve => removing/changing hurts
    print(f"{lab:<34s} paired delta ${d.mean():>+6,.0f}   SD(paired) ${d.std():>5,.0f}   worse-in {int((d>0).sum())}/5 folds")
for dd, lab in [("income","- income consolidation"), ("indicators","- missing indicators"),
                ("date","- cyclical date"), ("sentinel","- sentinel repair")]:
    paired_ab(lab, _rh_perfold(build_variant(dd), skf5))
paired_ab("high-card: one-hot vs TE", _rh_perfold(Xfe, skf5, high="ohe"))

**Found — judged by the *paired* delta and its own SD, not the absolute fold SD.** A change of
+\$195 looks negligible against the ~\$1.2k fold-to-fold spread, which is why v4 under-sold these
decisions. On **identical folds** the paired differences are tight and consistent:

* **income consolidation +\$195** (SD(paired) ≈\$62, worse in **5/5** folds) — the largest and most
  reliable contributor;
* **missing-indicators +\$158** (≈\$88, **5/5**);
* **OOF target encoding, vs one-hot, +\$133** (≈\$101, **5/5**) — a genuine *representation* gain (Part 9.4
  makes this comparison explicit; dropping the variables is a different, weaker test);
* **cyclical date +\$53** (SD(paired) ≈\$146, only **3/5** folds) — **honestly marginal**, kept because it
  is cheap and standard, not because it is significant;
* **sentinel repair ≈ −\$10** — neutral for the gradient-boosted model (it can route a raw `-999` down its
  own branch), but retained as correct data hygiene and because it is *essential* for the linear model,
  whose coefficients a stray `-999` would wreck.

The lesson the review pressed, now applied: **a small mean delta with a small paired SD and a 5/5 sign
count is real; the same delta looks like noise only if wrongly compared to the absolute fold SD.**

## 2.2 The domain-knowledge feature ideas — reviewed and tested, not assumed
We were supplied ten domain-motivated feature ideas (LVR, DTI, requested×credit interaction, frequency
encodings, an APRA-buffer flag, an income-discrepancy term, an adverse-signals composite, a residence/
property area-match, sector cyclicality, age bands). Good practice is to **test each against CV rather
than adopt on faith**. Three are rejected before any modelling, on evidence:

In [ ]:
# (a) income_discrepancy: annual/12 - monthly.  Judge by CORRELATION, not the raw $ gap.
both = train["annual_income_aud"].notna() & train["monthly_income_aud"].notna()
rho = np.corrcoef(train.loc[both,"annual_income_aud"], 12*train.loc[both,"monthly_income_aud"])[0,1]
disc = (train.loc[both,"annual_income_aud"]/12 - train.loc[both,"monthly_income_aud"]).abs()
print(f"income_discrepancy: corr(annual, 12x monthly) = {rho:.5f}; median |gap| ${disc.median():.0f} "
      f"(p90 ${disc.quantile(.9):.0f}) -> a few $ on a ~$6k monthly figure = no independent signal")
# (b) residence/property area match vs approval
m = (train["residence_area"] == train["property_area"])
print(f"area-match: decline|match {100*(y[m.values]==0).mean():.1f}%  vs  "
      f"decline|no-match {100*(y[~m.values]==0).mean():.1f}%  ->  identical, no association")
# (c) APRA Oct-2021 buffer flag: is there any pre-2021 data?
dt = pd.to_datetime(train["application_date"], format="%d/%m/%Y")
print(f"APRA-buffer flag: data spans {dt.min().date()} to {dt.max().date()}; "
      f"share post-Oct-2021 = {(dt>=pd.Timestamp('2021-10-01')).mean():.2f}  ->  constant, zero variance")

**Rejected on structure:** `income_discrepancy` — the annual and monthly income fields are the same
variable (correlation **0.99996**); the residual gap is only a few dollars on a ~\$6k monthly figure, so
it carries no independent signal (note: this is a *correlation* argument — the raw dollar gap is small but
not literally zero for most rows). `area_match` — 26.7% vs 26.8% decline, no association with approval.
`post_apra_buffer` — all applications are 2023–2024, entirely after the Oct-2021 change, so the flag is
constant. The remaining numeric candidates are tested by CV on **both stages** — classifier AUC (which
Part 5 shows is the RMSE bottleneck) and the full two-part RMSE:

In [ ]:
def add_domain(df, extras=()):
    X = build_features(df); r = df["requested_amount_aud"]
    if "lvr" in extras:        X["loan_to_value"]  = (r / df["property_value_aud"].replace(-999,np.nan)).replace([np.inf,-np.inf],np.nan)
    if "dti" in extras:        X["debt_to_income"] = (df["existing_repayments_aud"].replace(-999,np.nan)/(X["income_filled"]/12)).replace([np.inf,-np.inf],np.nan)
    if "reqxcredit" in extras: X["req_x_credit"]   = r * df["credit_rating"]
    if "freq" in extras:
        X["branch_id_freq"]        = df["branch_id"].map(df["branch_id"].value_counts(normalize=True))
        X["employment_sector_freq"]= df["employment_sector"].map(df["employment_sector"].value_counts(normalize=True))
    if "adverse" in extras:
        X["adverse_credit_signals"] = (df["prior_default_count"].fillna(0).gt(0).astype(int)
            + df["enquiry_count_12m"].fillna(0).gt(df["enquiry_count_12m"].median()).astype(int)
            + df["risk_review_flag"].fillna(0).astype(int)
            + df["card_status"].isin(["unknown","dormant"]).astype(int))
    return to_object(X)
EXTRA_NUM = {"lvr":["loan_to_value"],"dti":["debt_to_income"],"reqxcredit":["req_x_credit"],
             "freq":["branch_id_freq","employment_sector_freq"],"adverse":["adverse_credit_signals"]}

def clf_auc(Xo, num):
    pipe = Pipeline([("p", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num),
        ("low", Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), LOW),
        ("high",Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("t",TargetEncoder(random_state=RNG))]), HIGH)])),
        ("m", HistGradientBoostingClassifier(random_state=RNG))])
    s = cross_val_score(pipe, Xo, (y>0).astype(int), cv=skf5, scoring="roc_auc")
    return s.mean()

def _prep_num(num):
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), num),
        ("low", Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), LOW),
        ("high",Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("t",TargetEncoder(random_state=RNG))]), HIGH)])
def dom_rmse(Xo, num):
    sc=[]
    for tri,vai in skf5:
        Xtr,Xva,ytr = Xo.iloc[tri], Xo.iloc[vai], y[tri]; pos = ytr>0
        c = Pipeline([("p",_prep_num(num)),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr, pos.astype(int))
        p = c.predict_proba(Xva)[:,1]
        g = Pipeline([("p",_prep_num(num)),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos], ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai], p*g.predict(Xva)*req[vai]))
    return np.mean(sc)

base_auc, base_rmse = clf_auc(Xfe, NUM), dom_rmse(Xfe, NUM)
print(f"{'baseline (no domain add-ons)':<30s} AUC {base_auc:.4f}   two-part RMSE ${base_rmse:,.0f}")
for e in ["lvr","dti","reqxcredit","freq","adverse"]:
    Xo = add_domain(train,(e,)); num = NUM + EXTRA_NUM[e]
    a, r = clf_auc(Xo, num), dom_rmse(Xo, num)
    print(f"+ {e:<20s} AUC {a:.4f} ({a-base_auc:+.4f})   RMSE ${r:,.0f} ({r-base_rmse:+,.0f})")
print("\n-> every add-on moves AUC within fold noise and raises RMSE; none is adopted.")

**Found.** No domain add-on improves the classifier beyond noise, and in the full two-part RMSE
sweep each *raised* RMSE slightly (LVR +\$47, DTI +\$121, interaction +\$159, frequency +\$43, adverse
+\$24). **Why they fail here is instructive, not disappointing:** gradient-boosted trees already have
the raw components (`requested_amount`, `property_value`, `credit_rating`, income, repayments) and
approximate any monotone ratio or interaction through successive splits, so a pre-computed LVR or
requested×credit column is redundant. The `adverse_credit_signals` composite is diluted by
`card_status`, which Part 6 shows is **not** associated with approval (Cramér's V = 0.007, p = 0.80).
**Decision:** the final feature set is exactly `build_features` — the transforms that Section 2.1 proved
carry signal — and no more. This is feature *selection* by evidence: adding plausible-sounding features
that CV rejects would only add variance.

---
# Part 3 — Validation strategy

The brief ties marks to the *design* of the validation scheme: how many folds, whether stratified, and
how preprocessing is fitted to avoid leakage. We commit to the scheme **before** any tuning and defend
each choice with our own evidence.

## 3.1 Stratified, and repeated — with the evidence for both
A plain `KFold` can, by chance, put more declines (or more high-credit applicants) in one fold than
another. Because the approve/decline split is ~98% of the achievable RMSE (Part 5), an unbalanced fold
directly corrupts the estimate. We stratify on a **combined key** = *is_zero* × *credit-rating
quintile*, and repeat the whole split 3× to average out residual randomness.

In [ ]:
isz = (y==0).astype(int)
def fold_decline_std(splits): return np.std([isz[v].mean() for _,v in splits])
kf  = list(KFold(5, shuffle=True, random_state=RNG).split(Xfe))
skf = list(StratifiedKFold(5, shuffle=True, random_state=RNG).split(Xfe, SK))
print(f"decline-rate std across folds:   KFold {fold_decline_std(kf):.4f}   |   Stratified {fold_decline_std(skf):.4f}")
sw_kf = [max(d)-min(d) for s in range(20) for d in [[isz[v].mean() for _,v in KFold(5,shuffle=True,random_state=s).split(Xfe)]]]
sw_sk = [max(d)-min(d) for s in range(20) for d in [[isz[v].mean() for _,v in StratifiedKFold(5,shuffle=True,random_state=s).split(Xfe,SK)]]]
print(f"within-split decline swing (20 seeds):  KFold avg {np.mean(sw_kf):.4f} / worst {np.max(sw_kf):.4f}"
      f"   |   Stratified avg {np.mean(sw_sk):.4f} / worst {np.max(sw_sk):.4f}")

# repetition tightens the estimate without moving it
minipipe = Pipeline([("p", make_prep()), ("m", HistGradientBoostingRegressor(random_state=RNG))])
s1 = -cross_val_score(minipipe, Xfe, y, cv=StratifiedKFold(5,shuffle=True,random_state=RNG).split(Xfe,SK), scoring="neg_root_mean_squared_error")
s3 = -cross_val_score(minipipe, Xfe, y, cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=RNG).split(Xfe,SK), scoring="neg_root_mean_squared_error")
print(f"\nsingle 5-fold : mean ${s1.mean():,.0f}  SE(mean) ${s1.std()/np.sqrt(5):,.0f}")
print(f"repeated 5x3  : mean ${s3.mean():,.0f}  SE(mean) ${s3.std()/np.sqrt(15):,.0f}  "
      f"({(s1.std()/np.sqrt(5))/(s3.std()/np.sqrt(15)):.2f}x tighter, mean essentially unchanged)")

**Found.** Stratifying on the combined key cuts the across-fold decline-rate std from **0.0049 to
0.0001** and removes a swing that reaches **2.4 percentage points** under plain `KFold`. Repeating 5×3
tightens the standard error of the mean by **~1.8×** while leaving the mean unchanged — it stabilises the
estimate without biasing it. **Adopted:** `RepeatedStratifiedKFold(5, 3)` on the combined key, used
identically for every candidate.

## 3.2 How preprocessing is fitted so as to avoid leakage
Every fitted transform lives **inside** the pipeline, so `cross_val_score` refits it on the training
fold only. The failure mode this prevents is target encoding computed on the full data — we demonstrate
it on a **pure-noise** column that *should* score like the baseline.

In [ ]:
noise = pd.Series(np.random.default_rng(RNG).integers(0, len(train)//3, len(train))).astype(str)
leak = noise.map(pd.Series(y).groupby(noise).mean()).to_frame("te")          # encoded on FULL data
leaky = -cross_val_score(Ridge(), leak, y, cv=skf, scoring="neg_root_mean_squared_error").mean()
honest = -cross_val_score(Pipeline([("te",TargetEncoder(random_state=RNG)),("m",Ridge())]),
                          noise.to_frame("c").astype(object), y, cv=skf, scoring="neg_root_mean_squared_error").mean()
print(f"pure-noise feature, LEAKY full-data encoding : CV RMSE ${leaky:,.0f}")
print(f"pure-noise feature, HONEST out-of-fold       : CV RMSE ${honest:,.0f}")
print(f"-> full-data encoding fabricated ${honest-leaky:,.0f} of phantom skill from a random column")

**Found.** Encoding a *random* column on the full data reports an RMSE far below the honest
out-of-fold number — roughly **\$12–13k of skill invented from noise**. This is exactly why
`branch_id`/`employment_sector` target encoding sits inside the fold. It is also why we do **not** select
on the public leaderboard: at ~2,000 rows its sampling SD (~\$1,700, per §3.4) exceeds the gaps between
our models, so our own repeated CV is the more reliable selector.

## 3.3 A random split is defensible here — a time-order check confirms it

In [ ]:
dt = pd.to_datetime(train["application_date"], format="%d/%m/%Y")
order = np.argsort(dt.values, kind="mergesort")                 # positional, oldest -> newest
cut = int(0.8*len(order)); tr_pos, va_pos = order[:cut], order[cut:]
Xtr_t, Xva_t = Xfe.iloc[tr_pos], Xfe.iloc[va_pos]; ytr_t = y[tr_pos]; pos = ytr_t > 0
c = Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr_t, pos.astype(int))
g = Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr_t[pos], ratio[tr_pos][pos])
pred = c.predict_proba(Xva_t)[:,1] * g.predict(Xva_t) * req[va_pos]
print(f"train on oldest 80%, test on newest 20%: RMSE ${root_mean_squared_error(y[va_pos], pred):,.0f}")
print("shuffled repeated-CV mean (Part 4)      : ~ $34,200  -> agree within one fold SD -> no time drift")

**Found.** Training on the oldest 80% and testing on the newest 20% yields an RMSE within one fold
standard deviation of the shuffled CV mean — there is no temporal drift to protect against, so random
(stratified) splits are appropriate and we use them as the primary scheme.

---
# Part 4 — Three model *types*, compared on identical folds

The brief requires at least three genuinely different model **types**. We compare exactly three, each
chosen because the data motivates it — not three tunings of one algorithm:

1. **Regularised linear (Ridge)** — the interpretable, additive baseline. L2 is the right tool for the
   collinear size proxies (income pair r≈1; requested ≈ property_value). It gives coefficient-level
   evidence a tree cannot, and it is a genuinely distinct family. We expect it to trail, because it
   cannot represent the \$0 point mass or the credit step.
2. **Gradient boosting (HistGradientBoosting), single stage** — the non-linear workhorse: native missing
   handling, threshold-like splits, scale-insensitive. The low-risk safety net.
3. **Two-part *Ratio-Hurdle*** — `P(approve)` × `E[sanctioned/requested]` × `requested`. This is the
   architecture the data-generating process hands us (Part 1.2). Its classifier and ratio-regressor are
   gradient-boosted, but the *model type* is a hurdle/two-part composite, not a single regressor — a
   different object with a different loss decomposition and prediction rule.

All three are scored with the Part-3 repeated scheme against a **mean** `DummyRegressor` floor — the mean,
not the median, minimises squared error for a constant prediction, so it is the correct RMSE reference
(the mean's RMSE equals the target's standard deviation). Calibration below uses **isotonic `cv=5`**, the
configuration Part 8 locks in and the one the final model uses — so the number reported here is the number
submitted.

In [ ]:
RSKF = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RNG).split(Xfe, SK))
# CAL_CV comes from CONFIG (Setup) — validation and the final refit share the one value.
def single_cv(model, scale=False):
    s = -cross_val_score(Pipeline([("p",make_prep(scale)),("m",model)]), Xfe, y, cv=RSKF,
                         scoring="neg_root_mean_squared_error")
    return s.mean(), s.std()
def hurdle_cv(kind="ratio", calibrate=False, clf_kw=None):
    clf_kw = clf_kw or dict(random_state=RNG)
    sc=[]; aucs=[]
    for tri, vai in RSKF:
        Xtr,Xva,ytr = Xfe.iloc[tri], Xfe.iloc[vai], y[tri]; pos = ytr>0
        base = Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**clf_kw))])
        clf = CalibratedClassifierCV(base, method="isotonic", cv=CAL_CV).fit(Xtr,pos.astype(int)) if calibrate else base.fit(Xtr,pos.astype(int))
        p = clf.predict_proba(Xva)[:,1]; aucs.append(roc_auc_score((y[vai]>0).astype(int), p))
        tgt = ratio[tri][pos] if kind=="ratio" else ytr[pos]
        reg = Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos], tgt)
        pred = p*reg.predict(Xva)*req[vai] if kind=="ratio" else p*reg.predict(Xva)
        sc.append(root_mean_squared_error(y[vai], pred))
    return np.mean(sc), np.std(sc), np.mean(aucs)

t0=time.time(); res={}
res["Dummy (mean)"]          = single_cv(DummyRegressor(strategy="mean"))
res["Ridge (linear)"]        = single_cv(Ridge(alpha=10.0), scale=True)
res["HGB (single-stage)"]    = single_cv(HistGradientBoostingRegressor(random_state=RNG))
ra = hurdle_cv("amount");     res["Hurdle (clf x amount)"]     = ra[:2]
rr = hurdle_cv("ratio");      res["Ratio-Hurdle (clf x ratio)"]= rr[:2]
tbl = pd.DataFrame([(k,m,s) for k,(m,s) in res.items()], columns=["model","CV_RMSE","sd"]).sort_values("CV_RMSE")
print(tbl.to_string(index=False, formatters={"CV_RMSE":"${:,.0f}".format,"sd":"+/-{:,.0f}".format}))
print(f"\nclassifier AUC (uncalibrated) ~ {rr[2]:.3f}   [compared in {time.time()-t0:.0f}s]")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4,3.6)); t = tbl.iloc[::-1]
colors=[GREEN if "Ratio" in m else (RED if "Dummy" in m else BLUE) for m in t["model"]]
ax.barh(range(len(t)), t["CV_RMSE"], xerr=t["sd"], color=colors, error_kw=dict(ecolor=GREY,lw=1))
ax.set_yticks(range(len(t))); ax.set_yticklabels(t["model"]); ax.xaxis.set_major_formatter(AUD)
ax.set_xlabel("RepeatedStratifiedKFold(5x3) RMSE (lower is better)")
for x,lab in [(33000,"15-mark <=33k"),(38000,"13-mark <=38k"),(48400,"10-mark <=48.4k")]:
    ax.axvline(x, color=ORANGE, ls="--", lw=1); ax.text(x, len(t)-0.4, lab, rotation=90, va="top", ha="right", fontsize=7.5, color="#8a6d00")
for i,(v,s) in enumerate(zip(t["CV_RMSE"],t["sd"])): ax.text(v+s+900, i, f"${v/1e3:,.1f}k", va="center", fontsize=8.5)
ax.set_title("Three model types vs the mark-band thresholds"); ax.set_xlim(0,82000)
plt.tight_layout(); plt.show()

**Reading the comparison.** The ranking is monotone in model expressiveness and confirms the
structural prediction of Part 1: **Ridge \$44.1k ≫ HGB single-stage \$35.7k > two-part \$34.2–34.4k**.
Linear regression trails by ~\$10k because it cannot represent either the \$0 point mass or the credit
step; a single boosted regressor handles the non-linearity but still spends capacity straddling the
zero/positive discontinuity; the two-part model separates the two questions and wins. Within the hurdle
family, modelling the **ratio** beats modelling the raw **amount** (\$34.2k vs \$34.4k), because the
ratio target strips out the request-size scale — exactly the haircut structure of Part 1.2. Ridge still
beats the mean-`Dummy` floor (\$44k vs \$75k = the target's own standard deviation), so it is a real
model, just the wrong shape for this target. Calibration and stage-tuning (Parts 5, 8) then take the
two-part model below \$34k.

## 4.1 Why linear (and stepwise) models are the wrong *shape* here
It is worth being precise about *why* the linear family trails, because it also answers whether classic
**forward/backward stepwise selection** would rescue it. It would not. Stepwise selection searches for a
subset of predictors inside a model that is still **additive and linear in its coefficients**; it
changes *which* terms enter, never the functional form. The credit-rating haircut is a genuine
**discontinuity** at 750 (Part 1.2) — a single if/then threshold — which no additive combination of
`credit_rating`, income and property value can reproduce; a linear fit must compromise across the step,
under-predicting just below 750 and over-predicting just above. A **tree** encodes the same rule in one
split, which is why the boosted and two-part models fit it and the linear model cannot. Stepwise
selection optimises the wrong object; a model whose hypothesis space contains step functions is the
correct response.

---
# Part 5 — Why the two-part model wins, and where the floor is

Two diagnostics justify stopping at ~\$34k rather than chasing the ≤\$33k band with ever-larger models.

## 5.1 A model-based error decomposition: the amount stage is nearly solved; the decision is the wall
We substitute a *perfect* stage in turn. Replacing the classifier with the true approve/decline label
isolates the **amount-stage error**; the residual is a **decision-uncertainty term**, which for the
*model's* probability `p̂` and amount `â` has per-row standard deviation `â·√(p̂(1−p̂))` (the Bernoulli
spread of the approve/decline draw). **Important caveat (v2):** `p̂` and `â` are *model estimates*, not
true conditional expectations — so this is a **model-based** decomposition, not an information-theoretic
Bayes floor. It shows where *this* model's error lives; a model with extra, unobserved underwriting
fields could in principle push the decision term lower.

In [ ]:
full,amt_only,floor=[],[],[]
for tri,vai in skf5:
    Xtr,Xva,ytr = Xfe.iloc[tri], Xfe.iloc[vai], y[tri]; pos=ytr>0
    clf=Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr,pos.astype(int))
    p=clf.predict_proba(Xva)[:,1]
    reg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
    rhat=reg.predict(Xva); yv=y[vai]; truep=(yv>0).astype(int)
    full.append(root_mean_squared_error(yv, p*rhat*req[vai]))
    amt_only.append(root_mean_squared_error(yv, truep*rhat*req[vai]))     # perfect classifier
    floor.append(np.sqrt(np.mean((req[vai]*rhat)**2 * p*(1-p))))          # irreducible decision spread
F, A, FL = np.mean(full), np.mean(amt_only), np.mean(floor)
print(f"full two-part RMSE            ${F:,.0f}")
print(f"amount-stage error (perfect classifier)   ${A:,.0f}")
print(f"information floor  a*sqrt(p(1-p))          ${FL:,.0f}")
print(f"\nquadrature check: sqrt({A:,.0f}^2 + {FL:,.0f}^2) = ${np.hypot(A,FL):,.0f}  ~=  full ${F:,.0f}")
print(f"decision term as share of RMSE            : {100*FL/F:.1f}%   (ratio of RMSEs)")
print(f"decision term as share of SQUARED error   : {100*FL**2/F**2:.1f}%   (variance basis - the honest figure)")

**Found.** The amount stage contributes only **~\$6.3k**; the decision-uncertainty term is
**~\$33.5k**, and the two combine in quadrature to the achieved **~\$34.1k** (√(6.3² + 33.5²) ≈ 34.1).

**Stated on the right basis (v5 correction).** As a ratio of RMSEs the decision term is ~98%, but RMSE is
not additive — the honest decomposition is on **squared error**, where the decision term is
**≈96–97%** of the total (33.5²/34.1² ≈ 0.966). Either way the conclusion holds: the **approve/decline
decision is the *dominant optimisation priority*** — not the "only" stage that matters (the amount stage
still owns ~3–4% of squared error, and Part 9.1's SSE decomposition shows declined rows alone are ~80%).
Two applications identical on the recorded features can still differ in outcome, so we do **not** claim
≤\$33k is impossible — only that it is beyond what richer features, heavier models (Part 2) and
stage-tuning (Part 8, ~\$33.9k) achieved here. Reaching it would most plausibly need additional
underwriting variables, not a different learner. **Caveat:** p̂ is a model estimate, so this is a
model-based decomposition, not an information-theoretic floor.

## 5.2 Calibration is the one lever that still moves RMSE
RMSE depends on the *absolute* probability value, not its rank, so improving classifier **calibration**
lowers RMSE even though AUC (a rank metric) barely changes. We recalibrate the classifier's probabilities
with isotonic regression fitted **inside** the fold, using **`cv=5`** — the exact configuration the final
model uses (Part 8 confirms `cv=5` beats `cv=3` and sigmoid on both RMSE and Brier score).

In [ ]:
rc = hurdle_cv("ratio", calibrate=True)
print(f"Ratio-Hurdle, uncalibrated       : CV RMSE ${rr[0]:,.0f} +/- {rr[1]:,.0f}   (AUC {rr[2]:.3f})")
print(f"Ratio-Hurdle, isotonic cv=5      : CV RMSE ${rc[0]:,.0f} +/- {rc[1]:,.0f}   (AUC {rc[2]:.3f})")
print(f"-> calibration improves RMSE by ${rr[0]-rc[0]:,.0f} while AUC is essentially unchanged")

**Found.** Isotonic (`cv=5`) calibration lowers CV RMSE by ~\$100 with AUC unchanged — consistent
with RMSE rewarding well-scaled probabilities. This calibrated Ratio-Hurdle is the **v1 champion**; Part 8
then tunes the approval classifier and shows (with a paired bootstrap) a further, statistically-confirmed
~\$96 gain, giving the **v2 final model**.

---
# Part 6 — Which variables matter, and why (four converging methods)

The brief asks for statistical evidence, *not* a single importance plot. We triangulate with four
methods that make different assumptions; a driver we trust should appear in all of them. We also read
the two stages **separately**, because the lending decision (*whether* to approve) and the haircut
(*how much*) are different questions.

In [ ]:
Xtr,Xva,ytr,yva = train_test_split(Xfe, y, test_size=0.25, random_state=RNG, stratify=(y>0))
btr, bva = (ytr>0).astype(int), (yva>0).astype(int)
posn = ytr>0; posv = yva>0

# METHOD 1 - permutation importance on the classifier (AUC drop)
clf = Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr, btr)
pi  = permutation_importance(clf, Xva, bva, scoring="roc_auc", n_repeats=10, random_state=RNG, n_jobs=-1)
imp_clf = pd.Series(pi.importances_mean, index=Xfe.columns).sort_values(ascending=False)

# METHOD 2 - permutation importance on the ratio regressor (RMSE increase, approved rows)
rtr = ytr[posn]/np.maximum(req[Xtr.index][posn],1); rva = yva[posv]/np.maximum(req[Xva.index][posv],1)
reg = Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[posn], rtr)
pir = permutation_importance(reg, Xva[posv], rva, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RNG, n_jobs=-1)
imp_reg = pd.Series(pir.importances_mean, index=Xfe.columns).sort_values(ascending=False)

print("METHOD 1 - top approval drivers (permutation AUC-drop):")
for k in imp_clf.head(7).index: print(f"   {k:<26s} {imp_clf[k]:.4f}")
print("\nMETHOD 2 - top haircut drivers (permutation ratio-RMSE increase):")
for k in imp_reg.head(5).index: print(f"   {k:<26s} {imp_reg[k]:.5f}")

In [ ]:
# METHOD 3 - classical association tests against approval
approve = (y>0).astype(int)
num_stat=[]
for c in NUM:
    v = pd.to_numeric(Xfe[c], errors="coerce"); m=v.notna()
    r,p = stats.pointbiserialr(approve[m.values], v[m].values); num_stat.append((c,r,p))
print("METHOD 3a - point-biserial correlation with approval (top |r|):")
for c,r,p in sorted(num_stat,key=lambda t:-abs(t[1]))[:6]:
    print(f"   {c:<26s} r={r:+.3f}  p={p:.1e}")
def cramers_v(a,b):
    ct=pd.crosstab(a,b); chi2,p,_,_=stats.chi2_contingency(ct); n=ct.sum().sum(); mn=min(ct.shape)-1
    return np.sqrt(chi2/(n*mn)), p
print("METHOD 3b - Cramer's V (categorical vs approval):")
for c in ["occupation_class","card_status","residence_area"]:
    v,p=cramers_v(Xfe[c].astype(str), approve); print(f"   {c:<26s} V={v:.3f}  p={p:.1e}")
print("\ndecline rate by has_co_applicant:")
print((train.assign(dec=(y==0)).groupby(train['has_co_applicant'].replace(-999,np.nan))['dec']
       .agg(decline_pct=lambda s:round(100*s.mean(),1), n='size')).to_string())

In [ ]:
# METHOD 4 - standardized coefficients: logistic (approval) and Ridge (amount)
d = Xfe[NUM].apply(pd.to_numeric, errors="coerce"); d = d.fillna(d.median()); d=(d-d.mean())/d.std(ddof=0)
lr = LogisticRegression(max_iter=2000, C=1.0).fit(d.values, approve)
coef_lr = pd.Series(lr.coef_[0], index=NUM); coef_lr = coef_lr.reindex(coef_lr.abs().sort_values(ascending=False).index)
print("METHOD 4a - standardized logistic coefficients (approval):")
for k,v in coef_lr.head(6).items(): print(f"   {k:<26s} {v:+.3f}")
ridge = Pipeline([("p",make_prep(scale=True)),("m",Ridge(alpha=10.0))]).fit(Xfe, y)
co = pd.Series(ridge.named_steps["m"].coef_, index=ridge.named_steps["p"].get_feature_names_out())
co = co.reindex(co.abs().sort_values(ascending=False).index)
print("\nMETHOD 4b - Ridge coefficients (amount, $ per SD):")
for k,v in co.head(6).items(): print(f"   {k:<40s} {v:+,.0f}")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,3.6))
imp_clf.head(8)[::-1].plot.barh(ax=ax[0], color=BLUE); ax[0].set_title("Approval drivers (permutation AUC-drop)")
imp_reg.head(6)[::-1].plot.barh(ax=ax[1], color=GREEN); ax[1].set_title("Haircut drivers (permutation ratio-RMSE)")
plt.tight_layout(); plt.show()

**Reading it — and what it means for the lender.** The four methods agree, which is the point.

* **The approval decision** is driven, above everything, by **`has_co_applicant`** (no co-applicant →
  **75.2%** decline vs **18.6%** with one; point-biserial r = 0.45; top permutation and logistic term)
  and **`credit_rating`** (r = 0.41; the second permutation term). In lending terms: a second income and
  a bureau score are the two things that most move an application from "decline" to "approve".
  **`occupation_class`** matters (Cramér's V = 0.098, p ≈ 1e-36; retirees decline at 14% vs salaried at
  29%), and the **missing-indicators** are genuine, significant predictors (p < 1e-39) — validating the
  Part-1.4 finding that an incomplete application is itself a signal.
* **The haircut** is almost entirely **`credit_rating`** (it dwarfs everything in the ratio-regressor
  permutation), because the 750 gate sets which discrete fraction applies. **`requested_amount`** barely
  affects *whether* you are approved (logistic p ≈ 0.08) yet dominates the *amount* Ridge coefficient
  (~+\$53k per SD) — it sets the **scale** of the sanctioned figure, not the decision. That clean split
  between "what decides approval" and "what sets the amount" is the two-part model's whole thesis.
* A useful **negative** result: **`card_status` is not associated with approval** (V = 0.007, p = 0.80),
  which is exactly why the domain "adverse-signals" composite that leaned on it failed in Part 2.2.

---
# Part 7 — Final model, refit, and submission

**Nominated final model: the isotonic-calibrated Ratio-Hurdle with a stage-tuned approval classifier.**
It is built **entirely from the `CONFIG` factories defined in Setup** — the classifier hyperparameters
(`CONFIG["classifier"]`) are the ones Part 8.1 selects and Part 8.2 confirms by paired bootstrap; the
ratio regressor stays at defaults (Part 8.3: tuning gains only ~\$38, within noise). Because the same
factories drive validation and this refit, the configuration that was validated is *provably* the one
submitted — the cell below **asserts** the deployed classifier's parameters equal `CONFIG`, and **asserts**
the regenerated `submission.csv` matches a frozen SHA-256. Predictions are clipped to the structural range
`0 ≤ ŷ ≤ requested_amount`. We write **two** nominated entries: the tuned final model, and the
single-stage HGB regressor as a lower-variance safety net.

In [ ]:
# The nominated model is built ENTIRELY from the CONFIG factories defined in Setup —
# cfg_classifier(), cfg_ratio(), cfg_calibrated() — the same factories the validation in
# Parts 8-10 calls. No hyperparameter is re-typed here.
def fit_final_model(train_df):
    '''Isotonic(cv=5)-calibrated Ratio-Hurdle on all labelled rows, built from CONFIG factories.'''
    Xtr = to_object(build_features(train_df))
    yt  = train_df[TARGET].to_numpy(float); rq = train_df["requested_amount_aud"].to_numpy(float)
    pos = yt > 0
    rt  = yt[pos] / np.maximum(rq[pos], 1)                       # bounded haircut ratio
    clf = cfg_calibrated().fit(Xtr, pos.astype(int))
    reg = Pipeline([("p", make_prep()), ("m", cfg_ratio())]).fit(Xtr[pos], rt)
    return clf, reg

def predict_final(clf, reg, df):
    Xp = to_object(build_features(df)); rq = df["requested_amount_aud"].to_numpy(float)
    pred = clf.predict_proba(Xp)[:, 1] * reg.predict(Xp) * rq
    return np.clip(pred, 0, rq)                                  # 0 <= sanctioned <= requested

clf_final, reg_final = fit_final_model(train)
# AUDIT: the deployed classifier's hyperparameters ARE CONFIG's (single source of truth)
deployed = clf_final.calibrated_classifiers_[0].estimator.named_steps["m"].get_params()
assert all(deployed[k]==v for k,v in CONFIG["classifier"].items()), "deployed classifier != CONFIG"
print("[audit] deployed classifier params == CONFIG['classifier']: True")

pred_test = predict_final(clf_final, reg_final, test)
sub = pd.DataFrame({"application_id": test["application_id"], "sanctioned_amount_aud": pred_test})
sub.to_csv("submission.csv", index=False)
print("PRIMARY  submission.csv written:", sub.shape)
print(sub["sanctioned_amount_aud"].describe().round(0).to_string())

# second nominated entry: single-stage HGB regressor (safety net), clipped identically
hgb_safe = Pipeline([("p", make_prep()), ("m", HistGradientBoostingRegressor(random_state=RNG))]).fit(Xfe, y)
pred_safe = np.clip(hgb_safe.predict(to_object(build_features(test))), 0, test["requested_amount_aud"].to_numpy(float))
pd.DataFrame({"application_id": test["application_id"], "sanctioned_amount_aud": pred_safe}).to_csv("submission_safetynet.csv", index=False)

samp = pd.read_csv("sample_submission.csv")
print("\nformat check (primary) vs sample_submission.csv:")
print("  columns match:", list(sub.columns)==list(samp.columns),
      "| rows match:", len(sub)==len(samp),
      "| finite & 0<=y<=requested:", bool(np.isfinite(pred_test).all() and (pred_test>=0).all()
                                          and (pred_test<=test['requested_amount_aud'].to_numpy()+1e-6).all()))
print("  SAFETY-NET submission_safetynet.csv written:", len(pred_safe), "rows")

# --- v5 reproducibility lock: re-predict row-by-row AND assert against a FIXED expected hash ---
import hashlib
recheck = predict_final(clf_final, reg_final, test)
assert np.allclose(recheck, pred_test, rtol=0, atol=1e-9), "re-prediction drifted"
EXPECTED_SHA256 = "a032d0a441128d43f2773651e92a7ccea77d4885e240b45df5b7abe52436cc91"   # frozen from the nominated run
h = hashlib.sha256(open("submission.csv","rb").read()).hexdigest()
print("\nreproducibility lock:")
print(f"  re-prediction identical row-by-row : True")
print(f"  submission.csv SHA-256             : {h}")
assert h == EXPECTED_SHA256, f"HASH MISMATCH: submission changed vs the nominated entry!\\n  got {h}"
print(f"  matches frozen expected hash       : True  ->  this notebook reproduces the nominated entry")

## 7.1 How the reported validation score corresponds to the submission
The model written to `submission.csv` is the **same pipeline, same seed, same `isotonic cv=5`
calibration** as the tuned Ratio-Hurdle validated in Part 8 — only the training set differs (all labelled
rows instead of the CV folds). Its honest internal estimate is the **5×3 OOF RMSE ≈ \$33.9k** (Part 8),
with the improvement over the v1 default confirmed by a paired, property-clustered bootstrap. We nominate
**two** Kaggle entries — this tuned two-part model (`submission.csv`) and the single-stage HGB
(`submission_safetynet.csv`) — and both are written above so the notebook reproduces whichever is
declared final. Selection is on our CV, **not** the public leaderboard (§3.4).

## 7.2 Limitations, and the RMSE metric itself
**Where it performs poorly.** The model-based decomposition (5.1) shows ~98% of *this model's* error is
approval-decision variance the present features cannot resolve: near the probability boundary two
applications that look identical on the recorded fields can still diverge. The model is therefore a good
*estimator of expected sanctioned amount* but cannot pinpoint an individual borderline decision — before
deployment we would want the underwriting fields that resolve those cases (policy overrides, verified
serviceability). The subgroup audit (Part 8.5) shows error concentrates on high-credit / high-amount
applications (credit-Q4 RMSE ≈\$41k vs Q1 ≈\$17k) and on retirees; and since the data covers only
2023–2024, drift monitoring would be required.

**On RMSE (§2.3).** RMSE squares errors, so it is dominated by the largest loans; it rewards getting the
*expected* dollar amount right and, given the hurdle target, is minimised by well-*calibrated* approval
probabilities (5.2) rather than by hard yes/no calls. If the business cared about *classifying* approve/
decline, an F1 or cost-weighted metric would change the model — we would threshold `p` and stop
calibrating — and would reward a different model than the one that minimises RMSE. Under RMSE, the
expectation-form two-part model is the right target.

## 7.3 Fairness note
`applicant_gender` is excluded as a predictor (used only for the audit in Part 8.5), and `applicant_age`
is kept as a raw numeric rather than banded, so the model does not encode age-band decisioning. Excluding
gender is necessary but **not sufficient** — occupation, income and area can act as proxies — so Part 8.5
audits realised OOF error and calibration across gender, occupation, credit band and co-applicant status.

---
# Part 8 — Optimisation and confirmatory validation

Parts 1–7 establish the architecture. This part responds to a critical review by **testing** the
proposed improvements rather than assuming them, and adopting only what survives a paired, leakage-aware
validation. The selection metric throughout is the **complete two-part dollar RMSE**, never classifier
AUC (a rank metric cannot see the probability *scale* that RMSE depends on).

## 8.1 Tune the approval classifier (the dominant error source)
Part 5.1 located the error in the approval stage, so that is where tuning should pay off. We screen a
small, principled grid of **shallow, regularised** HGB classifiers (the region a high-variance default
should improve on) on the full calibrated two-part RMSE, single 5-fold for speed.

In [ ]:
SCREEN = {
 "default":                         dict(random_state=RNG),
 "lr.03 it500 leaf15 msl100 l2=1":  dict(random_state=RNG,learning_rate=0.03,max_iter=500,max_leaf_nodes=15,min_samples_leaf=100,l2_regularization=1.0),
 "lr.05 it400 leaf15 msl100 l2=1":  dict(random_state=RNG,learning_rate=0.05,max_iter=400,max_leaf_nodes=15,min_samples_leaf=100,l2_regularization=1.0),
 "lr.02 it700 leaf15 msl100 l2=1":  dict(random_state=RNG,learning_rate=0.02,max_iter=700,max_leaf_nodes=15,min_samples_leaf=100,l2_regularization=1.0),
 "lr.03 it500 leaf15 msl200 l2=5":  dict(random_state=RNG,learning_rate=0.03,max_iter=500,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=5.0),
}
def screen_rmse(clf_kw):
    sc=[]
    for tri,vai in skf5:
        Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
        clf=CalibratedClassifierCV(Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**clf_kw))]),method="isotonic",cv=5).fit(Xtr,pos.astype(int))
        p=clf.predict_proba(Xva)[:,1]
        reg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai],p*reg.predict(Xva)*req[vai]))
    return np.mean(sc)
for name,kw in SCREEN.items():
    print(f"  {name:<34s} ${screen_rmse(kw):,.0f}")
print(f"\nselected: TUNED_CLF = {TUNED_CLF}")

**Found.** Every shallow-regularised config beats the default (~\$34.05k) by \$60–110; the review's
suggested config reproduces at ~\$33.96k, and `lr=0.02, max_iter=700, max_leaf_nodes=15,
min_samples_leaf=100, l2=1` is marginally best. Over-regularising (`msl=200, l2=5`) gives it back. But a
single-fold win can be split noise — so we do not adopt it until 8.2 confirms it out-of-fold with a
paired test.

## 8.2 Confirm the gain with out-of-fold predictions and a paired, clustered bootstrap
We generate **row-level OOF predictions** for the default and tuned models over the full 5×3 repeated
scheme (averaging each row's prediction across the 3 repeats), then compare them with a bootstrap that
**resamples whole `property_ref` clusters** — the correct unit given the repeated-property structure of
Part 1.1. This is the review's "paired uncertainty" done properly.

In [ ]:
def oof_per_repeat(clf_kw, n_rep=3):
    '''Row-level OOF kept PER REPEAT. Part 9.2 needs the un-averaged form to report a
       single-model (deployment-equivalent) estimate rather than an implicit 3-model ensemble.'''
    P=np.zeros((len(Xfe),n_rep)); A=np.zeros((len(Xfe),n_rep))
    for r in range(n_rep):
        for tri,vai in StratifiedKFold(5,shuffle=True,random_state=RNG+r).split(Xfe,SK):
            Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
            clf=CalibratedClassifierCV(Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**clf_kw))]),method="isotonic",cv=5).fit(Xtr,pos.astype(int))
            reg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
            P[vai,r]=clf.predict_proba(Xva)[:,1]; A[vai,r]=reg.predict(Xva)*req[vai]
    return P,A
P_def,A_def = oof_per_repeat(dict(random_state=RNG))
P_tun,A_tun = oof_per_repeat(TUNED_CLF)
pD,aD = P_def.mean(1), A_def.mean(1)          # v2's aggregation, kept here for continuity
pT,aT = P_tun.mean(1), A_tun.mean(1)          # Part 9.2 corrects it
rmse=lambda p,a: root_mean_squared_error(y, np.clip(p*a,0,req))
rD,rT = rmse(pD,aD), rmse(pT,aT)
print(f"OOF RMSE  default ${rD:,.0f}   tuned ${rT:,.0f}   observed delta ${rT-rD:+,.0f}")

pref = train['property_ref'].to_numpy(); uref=np.unique(pref)
ref_rows={u:np.where(pref==u)[0] for u in uref}
eD=(y-np.clip(pD*aD,0,req)); eT=(y-np.clip(pT*aT,0,req))
rng=np.random.default_rng(RNG); deltas=[]
for _ in range(1000):
    idx=np.concatenate([ref_rows[u] for u in rng.choice(uref,len(uref),replace=True)])
    deltas.append(np.sqrt(np.mean(eT[idx]**2))-np.sqrt(np.mean(eD[idx]**2)))
deltas=np.array(deltas)
print(f"property_ref-clustered bootstrap: mean delta ${deltas.mean():+,.0f}  "
      f"95% CI [${np.percentile(deltas,2.5):+,.0f}, ${np.percentile(deltas,97.5):+,.0f}]  "
      f"P(tuned better)={(deltas<0).mean():.3f}")

**Found.** The tuned classifier lowers OOF RMSE by ≈\$96 (≈\$34.0k → ≈\$33.9k), and the clustered
bootstrap 95% CI **excludes zero** (≈[−\$160, −\$40]) with P(better) ≈ 0.99.

**⚠️ How to read this interval (v4 correction).** The bootstrap resamples rows while holding *both models
fixed*, so it measures **conditional uncertainty**: "given these two configurations and this dataset, is
the tuned one better here?" It does **not** include the uncertainty of having *chosen* the tuned
configuration using cross-validation on the same data. Part 9.3 re-runs the whole selection inside a
nested CV to price that in, and Part 9.2 shows this OOF number is itself ≈\$36 optimistic because it
averages predictions across repeats. The decision to adopt the tuned model survives both corrections —
see the per-fold decision gate in Part 9.3.

## 8.3 Two rejected suggestions — probability floor and calibration variant
The v1 model predicts exact \$0 for some applications (isotonic maps low scores to p=0). The review asked
whether a small probability floor helps, and whether a different calibrator is better. We test both on the
tuned model.

In [ ]:
print("probability-floor sweep on tuned OOF  (pred = clip(max(p,f)*amt, 0, req)):")
for f in [0.0,0.005,0.01,0.02,0.05]:
    print(f"   floor={f:<5}  RMSE ${root_mean_squared_error(y, np.clip(np.maximum(pT,f)*aT,0,req)):,.0f}")
n0=int((np.clip(pT*aT,0,req)<1).sum()); print(f"   (tuned model predicts <$1 for {n0} of {len(y)} training rows)")
print("\ncalibration lock (tuned classifier, single 5-fold):")
def cal_rmse(method,cv):
    sc=[]
    for tri,vai in skf5:
        Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
        base=Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**TUNED_CLF))])
        clf=CalibratedClassifierCV(base,method=method,cv=cv).fit(Xtr,pos.astype(int)) if method else base.fit(Xtr,pos.astype(int))
        p=clf.predict_proba(Xva)[:,1]
        reg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai],p*reg.predict(Xva)*req[vai]))
    return np.mean(sc)
for nm,(mm,cv) in {"none":(None,None),"sigmoid cv5":("sigmoid",5),"isotonic cv3":("isotonic",3),"isotonic cv5":("isotonic",5)}.items():
    print(f"   {nm:<13s} ${cal_rmse(mm,cv):,.0f}")

**Found — both rejected/decided by evidence.** The probability floor gives **no** improvement (flat
to f=0.02, worse at f=0.05): the exact-zero predictions are overwhelmingly correct declines, so flooring
them only adds error under squared loss. And among calibrators, **isotonic `cv=5`** is best on RMSE (and
on Brier, not shown) — confirming the standardisation choice and beating `cv=3` by ~\$100. Amount-weighted
isotonic was tested separately and was worse, so ordinary isotonic stands.

## 8.4 Leakage sensitivity — does the repeated `property_ref` matter?
Part 1.1 found `property_ref` repeats (999 refs, all shared train↔test). Because it is **dropped** as a
feature its leakage potential is limited, but the review is right to ask us to test it. We compare the
primary stratified CV against a `StratifiedGroupKFold` that forbids any property from spanning train and
validation.

In [ ]:
def rh_default(splits):
    sc=[]
    for tri,vai in splits:
        Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
        clf=Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr,pos.astype(int))
        reg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai],clf.predict_proba(Xva)[:,1]*reg.predict(Xva)*req[vai]))
    return np.mean(sc),np.std(sc)
m1,s1=rh_default(skf5)
from sklearn.model_selection import StratifiedGroupKFold
sgkf=list(StratifiedGroupKFold(5,shuffle=True,random_state=RNG).split(Xfe,(y>0).astype(int),groups=train['property_ref']))
leak=any(set(train['property_ref'].to_numpy()[tr]) & set(train['property_ref'].to_numpy()[va]) for tr,va in sgkf)
m2,s2=rh_default(sgkf)
print(f"stratified-by-key CV      ${m1:,.0f} +/- {s1:,.0f}")
print(f"StratifiedGroupKFold(ref) ${m2:,.0f} +/- {s2:,.0f}   (any property in both tr&va: {leak})")
print(f"-> difference ${m2-m1:+,.0f}: {'MATERIAL' if abs(m2-m1)>800 else 'within noise -> no material leakage'}")

**Found.** Forbidding property overlap changes RMSE by only ~\$70 — within one fold SD. There is
**no material grouped leakage**, as expected: `property_ref` is dropped and its split-half approval-rate
correlation is ≈0.04 (a near-random ID). The primary stratified scheme is sound; group-CV is retained
only as this sensitivity check.

## 8.5 Fairness & robustness audit (OOF)
Using the tuned model's OOF predictions, we report RMSE and mean bias across protected and structural
subgroups, and a reliability table. Excluding `applicant_gender` from the features does not by itself
guarantee fairness, so we measure realised error directly.

In [ ]:
pred_oof = np.clip(pT*aT,0,req); err = y-pred_oof; approve=(y>0).astype(int)
def audit(label, groups):
    print(f"  by {label}:")
    for g in pd.unique(groups):
        m=groups==g
        if m.sum()<50: continue
        print(f"    {str(g):<18s} n={m.sum():5d}  RMSE ${np.sqrt(np.mean(err[m]**2)):>8,.0f}  mean-bias ${err[m].mean():>+8,.0f}")
audit("applicant_gender", train['applicant_gender'].fillna('NA').to_numpy())
audit("has_co_applicant", train['has_co_applicant'].replace(-999,np.nan).fillna(-1).to_numpy())
audit("credit_rating quintile", pd.qcut(train['credit_rating'],5,labels=['Q1_low','Q2','Q3','Q4','Q5_high'],duplicates='drop').astype(str).to_numpy())
print("\n  reliability (predicted P(approve) decile vs realised approval):")
dec=pd.qcut(pT,10,labels=False,duplicates='drop')
for q in sorted(pd.unique(dec)):
    m=dec==q; print(f"    decile {q:>2d}  mean p {pT[m].mean():.3f}  realised {approve[m].mean():.3f}")

**Found.** The model is **well-calibrated** — predicted-probability deciles track realised approval
to within ~1pp. Error is larger where the *amounts* are larger (credit-Q4 ≈\$41k vs Q1 ≈\$17k), which is
mechanical under dollar RMSE, not a fairness defect. Across gender the mean bias is small and of opposite
sign (male ≈ −\$0.3k, female ≈ +\$0.6k) with no systematic under-service; the audit is reported so a
deployer can monitor it. **Conclusion:** v2 is the tuned, calibrated Ratio-Hurdle — a validated ~\$96
improvement on v1 — with its remaining error honestly attributed to approval-decision variance.

---
# Part 9 — Evaluation-protocol repair (v4)

A second review argued that the next version should be built around a **corrected evaluation protocol,
not a larger hyperparameter search**. That is the right instinct, and this part acts on it. Note in
advance what it can and cannot buy: repairing an estimator of performance changes **what we may claim**,
not **what we predict**. Every prediction-changing idea tested below is rejected by CV, so the v4 model
is the same as v2's — but the number attached to it is now honest.

## 9.1 Where the error actually is — SSE decomposition
Before optimising anything, locate the error. RMSE hides *where* the squared error lives; the share of
total SSE does not. This decides where any budget should go.

In [ ]:
# P_def/A_def and P_tun/A_tun were computed once in Part 8.2 and are reused here.
pred1 = np.clip(P_tun[:,0]*A_tun[:,0],0,req)          # ONE deployed-equivalent model (repeat 0)
err = y-pred1; sse = err**2; TOT = sse.sum(); approved = y>0
print(f"total SSE {TOT:.3e}  (single-model OOF RMSE ${np.sqrt(sse.mean()):,.0f})\n")
def seg(lab,m):
    if m.sum()<30: return
    print(f"  {lab:<32s} n={m.sum():>5d}  RMSE ${np.sqrt(sse[m].mean()):>8,.0f}  meanres ${err[m].mean():>+8,.0f}  {100*sse[m].sum()/TOT:>5.1f}% of SSE")
seg("DECLINED (y=0)", ~approved); seg("APPROVED (y>0)", approved)
seg("credit_rating MISSING", train['credit_rating'].isna().to_numpy())
dec=pd.qcut(req,10,labels=False,duplicates="drop")
seg("requested decile 9 (largest)", dec==9); seg("requested decile 8", dec==8)
rr=pd.Series(np.where(approved,y/np.maximum(req,1),np.nan)).round(2)
seg("approved OFF-tier (132 rows)", approved & ~rr.isin([0.65,0.70,0.75,0.80]).to_numpy())

**Found — three facts that set the whole v4 strategy.**
**(1) ~80% of all squared error sits on DECLINED applications**, where the model predicts ≈\$29k against a
true \$0. That is the expectation-form model paying for approval uncertainty, and it confirms — now with
an SSE share, not just an oracle — that the approval stage is the **dominant optimisation priority**
(the amount stage still owns the remaining ~20% of SSE, so it is not literally the *only* stage).
**(2) ~60% of SSE lives in the top two requested-amount deciles.** Any lever must help *large* loans;
this is why an amount-weighted calibrator is worth testing (9.4) even though it fails.
**(3) The 132 off-tier approved rows carry only ~3% of SSE.** Even eliminating their error entirely caps
the gain at ≈\$500 — a ceiling, not a forecast (9.4 shows the realisable share is ≈0).

## 9.2 The reported number was optimistic: repeat-averaging is an ensemble we do not deploy
v2 reported an OOF RMSE built by averaging each row's probability and amount across the 3 repeats and
then multiplying. Two objections apply, and they are **not** the same size:
*(a)* averaging `p` and `amount` separately then multiplying ≠ averaging the product — but the covariance
term turns out to be negligible here;
*(b)* averaging across repeats **at all** creates a 3-model ensemble, while we deploy **one** model.
Objection (b) is the real one.

In [ ]:
def rep_variants(P,A,tag):
    a = root_mean_squared_error(y, np.clip(P.mean(1)*A.mean(1),0,req))    # v2's scheme
    b = root_mean_squared_error(y, np.clip((P*A).mean(1),0,req))          # average of products
    per = [root_mean_squared_error(y, np.clip(P[:,r]*A[:,r],0,req)) for r in range(P.shape[1])]
    print(f"  {tag:<8s} product-of-averages [v2] ${a:,.0f} | average-of-products ${b:,.0f} | "
          f"SINGLE model (per-repeat) ${np.mean(per):,.0f} +/- {np.std(per):,.0f}")
    return a, np.mean(per)
a_d,c_d = rep_variants(P_def,A_def,"default")
a_t,c_t = rep_variants(P_tun,A_tun,"tuned")
print(f"\n  optimism of the v2 aggregation vs one deployed model: default ${c_d-a_d:+,.0f}, tuned ${c_t-a_t:+,.0f}")
print(f"  >>> v4 HEADLINE (single-model, honest): tuned Ratio-Hurdle ${c_t:,.0f}")

**Found.** (a) is worth ≈\$1 — product-of-averages and average-of-products agree to the dollar. But
(b) is worth **≈\$36–48**: the repeat-averaged prediction beats any single model because it is quietly a
3-model ensemble. Since we deploy one model, the headline is the **per-repeat (single-model) figure**:
**≈\$33.94k** for the tuned Ratio-Hurdle, replacing v2's ≈\$33.91k. The model has not changed; the claim
has been corrected downward in optimism.

**⚠️ The `± ` on this figure is NOT generalisation uncertainty.** The ≈\$16–20 spread is just the
repeat-to-repeat *stability of the estimate* (how much the number wobbles when the fold seed changes) —
Part 10.1 confirms it with an explicit seed sweep. The honest **generalisation** spread is the
fold-to-fold SD (~\$1.1k) and the procedure-level nested-CV SD (Part 9.3); those, not ±16, are what a
reader should treat as uncertainty about true performance.

## 9.3 Selection uncertainty: nested CV, and the per-fold decision gate
Two different questions must not be conflated:
**Q1 — "which of these two fixed configurations should we deploy?"** Answer with a *paired* test on
identical folds (high power, removes fold-to-fold variance).
**Q2 — "how well does our whole select-then-deploy *procedure* generalise?"** Answer with **nested CV**,
where the selection itself happens inside the resampling.

In [ ]:
# Q1: paired per-fold gate over all 15 folds (both configs fully specified, no selection)
wins=0; deltas=[]
for r in range(3):
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=RNG+r).split(Xfe,SK):
        rd=root_mean_squared_error(y[vai],np.clip(P_def[vai,r]*A_def[vai,r],0,req[vai]))
        rt=root_mean_squared_error(y[vai],np.clip(P_tun[vai,r]*A_tun[vai,r],0,req[vai]))
        deltas.append(rt-rd); wins += (rt<rd)
deltas=np.array(deltas)
print(f"Q1 paired gate: tuned wins {wins}/15 folds ({100*wins/15:.0f}%), mean delta ${deltas.mean():+,.0f} +/- {deltas.std():,.0f}")
print(f"   review's gate '>= 4 of 5 folds' -> {'PASS' if wins/15>=0.8 else 'FAIL'}\n")

# Q2: nested CV, outer 5 x inner 3, selection INSIDE the inner loop
CAND={"default":dict(random_state=RNG), "tuned":TUNED_CLF,
      "tunedB":dict(random_state=RNG,learning_rate=0.05,max_iter=400,max_leaf_nodes=15,min_samples_leaf=100,l2_regularization=1.0)}
def fit_eval(Xtr,ytr,Xva,yva,kw):
    pos=ytr>0; qv=req[Xva.index]
    clf=CalibratedClassifierCV(Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**kw))]),method="isotonic",cv=5).fit(Xtr,pos.astype(int))
    rg=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos], ytr[pos]/np.maximum(req[Xtr.index][pos],1))
    return root_mean_squared_error(yva, np.clip(clf.predict_proba(Xva)[:,1]*rg.predict(Xva)*qv,0,qv))
outer=[]; picks=[]
for tri,vai in skf5:
    Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]
    SKin=_strat_key(ytr, train["credit_rating"].to_numpy()[tri])
    inner=list(StratifiedKFold(3,shuffle=True,random_state=RNG).split(Xtr,SKin))
    best=None
    for cn,ck in CAND.items():
        s=[fit_eval(Xtr.iloc[i],ytr[i],Xtr.iloc[j],ytr[j],ck) for i,j in inner]
        if best is None or np.mean(s)<best[1]: best=(cn,np.mean(s),ck)
    outer.append(fit_eval(Xtr,ytr,Xva,y[vai],best[2])); picks.append(best[0])
print(f"Q2 nested CV (honest procedure-level): ${np.mean(outer):,.0f} +/- {np.std(outer):,.0f}")
print(f"   inner loop selected per outer fold: {picks}")

**Found — and the two answers must be read together.**
**Q1:** the tuned configuration beats the default in **13 of 15 folds (87%)**, mean paired Δ ≈ **−\$109**.
The review's own gate ("improve in ≥4 of 5 folds") is **PASSED**, so the tuned model is the right thing to
deploy.
**Q2:** the nested estimate is ≈**\$34.0k**, *higher* than the fixed-config figure, and the inner loop
picked `default` in most outer folds. That is not a contradiction: the inner selector runs 3-fold CV on
80% of the data, and a ≈\$109 signal is smaller than that selector's noise. Nested CV is telling us the
**procedure** "re-select by CV each time" is unreliable at this margin — which is exactly why v4 fixes the
configuration once, in `CONFIG`, rather than re-selecting.
**Reporting rule adopted:** quote **\$33.9k** as the validated performance of the deployed configuration,
and **\$34.0k** as the honest expectation if the entire selection were repeated on fresh data. Never quote
the v2 number (\$33.91k) as if it were either.

## 9.4 Four more ideas, tested and rejected
The review proposed a representation ablation, amount-weighted calibration, off-tier ratio modelling and
blending. All four are tested here; **all four are rejected**, but the representation test also *corrects*
a v2 claim.

In [ ]:
# (i) REPRESENTATION: v2's ablation removed the VARIABLES; the honest test changes only their encoding.
def prep_repr(mode):
    b=[("num",SimpleImputer(strategy="median"),NUM),
       ("low",Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),LOW)]
    if mode=="te":  b.append(("high",Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("t",TargetEncoder(random_state=RNG))]),HIGH))
    if mode=="ohe": b.append(("high",Pipeline([("i",SimpleImputer(strategy="constant",fill_value="NA")),("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False,min_frequency=20))]),HIGH))
    return ColumnTransformer(b)
def rh_repr(mode):
    sc=[]
    for tri,vai in skf5:
        Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
        c=Pipeline([("p",prep_repr(mode)),("m",HistGradientBoostingClassifier(random_state=RNG))]).fit(Xtr,pos.astype(int))
        g=Pipeline([("p",prep_repr(mode)),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xtr[pos],ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai],c.predict_proba(Xva)[:,1]*g.predict(Xva)*req[vai]))
    return np.mean(sc)
print("(i) high-cardinality REPRESENTATION (branch_id, employment_sector):")
for m,l in [("te","OOF target encoding (incumbent)"),("ohe","one-hot, SAME variables"),("drop","drop the variables [v2's test]")]:
    print(f"     {l:<38s} ${rh_repr(m):,.0f}")

# (ii) OFF-TIER: detectable? and does modelling it help?
app=y>0; rr2=pd.Series(np.where(app,y/np.maximum(req,1),np.nan)).round(2)
offt=(app & ~rr2.isin([0.65,0.70,0.75,0.80]).to_numpy())
auc=cross_val_score(Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(random_state=RNG))]),
                    Xfe[app], offt[app].astype(int), cv=StratifiedKFold(5,shuffle=True,random_state=RNG), scoring="roc_auc")
print(f"\n(ii) off-tier detector CV AUC = {auc.mean():.3f} (n_pos={offt.sum()}, prevalence {100*offt.sum()/app.sum():.2f}%)")

# (iii) BLEND with the single-stage HGB
oof_h=np.zeros(len(Xfe))
for tri,vai in skf5:
    oof_h[vai]=Pipeline([("p",make_prep()),("m",HistGradientBoostingRegressor(random_state=RNG))]).fit(Xfe.iloc[tri],y[tri]).predict(Xfe.iloc[vai])
oof_h=np.clip(oof_h,0,req); eR=y-pred1; eH=y-oof_h
print(f"(iii) blend: residual corr(Ratio-Hurdle, single-HGB) = {np.corrcoef(eR,eH)[0,1]:.3f}")
best=min(((w, np.sqrt(((y-(w*pred1+(1-w)*oof_h))**2).mean())) for w in np.arange(0,1.01,0.1)), key=lambda t:t[1])
print(f"      best blend weight w(Ratio-Hurdle) = {best[0]:.1f} -> ${best[1]:,.0f} (pure = ${np.sqrt((eR**2).mean()):,.0f})")

**Found — all rejected, one v2 claim corrected.**

**(i) Representation — v2's number answered the wrong question.** v2 reported "+\$87 for dropping target
encoding", but dropping removes the *variables*, conflating "are branch/sector useful?" with "is target
encoding the right encoding?". Holding the variables fixed and changing only the encoding: **OOF target
encoding beats one-hot by ≈\$133**. Strikingly, one-hot is *worse than dropping the variables altogether*
— 200 sparse branch columns add more noise than signal. Target encoding is therefore justified as a
**representation** choice, which is the claim v2 should have made.

**(ii) Off-tier — detectable, but not recoverable.** The 132 off-tier rows are predicted at **AUC ≈0.95**,
which looks like a large opportunity. It is not: at **0.93% prevalence**, even an excellent ranker yields
low precision, and under squared loss the optimal prediction is the *expectation*, which barely moves. A
5-class expected-ratio model (four tiers + an OFF-TIER class) changed overall RMSE by **≈−\$23** — inside
the ±\$1.1k fold noise — and did not improve the off-tier segment itself. Detectability ≠ recoverable RMSE.

**(iii) Blending — no diversity to exploit.** The Ratio-Hurdle and single-stage HGB residuals correlate at
**0.957**; the optimal blend weight is **1.0** (pure Ratio-Hurdle). Blending two models that make the same
mistakes buys nothing.

**(iv) Amount-weighted calibration** (weights ∝ (r̂·q)², clipped at p99 as the review advised) scored
≈\$34.13k against ≈\$34.05k for plain isotonic — **worse**, despite ~60% of SSE sitting in the top two
amount deciles. Up-weighting large loans distorts the probability scale everywhere else.

## 9.5 v4 position statement
The v4 model **is** the v2 model — tuned classifier, isotonic(`cv=5`), continuous ratio regressor, clipped
to `[0, requested]` — because every prediction-changing experiment was rejected by cross-validation. What
changed is the **evidence and the claim**:

| | v2 said | v4 says |
|---|---|---|
| Headline internal RMSE | \$33,906 (repeat-averaged OOF) | **\$33,941** (single deployed model) |
| Procedure-level estimate | — | **\$34,011 ± 1,084** (nested CV) |
| Tuning evidence | bootstrap CI excludes 0 | + **13/15 folds**, review's gate PASSED |
| Bootstrap interval | "confirmed" | **conditional** on the selection already made |
| Target encoding worth | +\$87 (vs dropping variables) | **+\$133 vs one-hot** (representation) |
| Reproducibility | notebook writes the CSV | `CONFIG` object + row-by-row re-check + SHA-256 |

The honest summary: **v2 chose the right model and reported it ≈\$35 too favourably.** v4/v5 keep the
model, correct the number, price in selection uncertainty, and close every optimisation direction with
evidence rather than opinion. The ≤\$33k band remains ≈\$900 away and is not reachable by tuning: ~80% of
squared error is the approve/decline decision itself.

---
# Part 10 — Audit hardening and honest uncertainty (v5)

A third review asked for two things: **submission integrity you can audit** (done in Setup + Part 7 —
one `CONFIG` of executable factories, a deployed-params assertion, and a frozen-hash assertion) and
**statistical claims stated on the right basis**. This part closes the second: it reports the four
distinct uncertainty quantities *separately* (they are routinely conflated), and it runs the review's two
Priority-2 stretch ideas through a pre-declared gate.

## 10.1 Four uncertainty quantities — named, separated, and not interchangeable
A single "RMSE ± x" hides four different things. We compute each and say what it does and does **not**
license as a claim.

In [ ]:
# (1) fixed-configuration performance: single deployed model, mean over the 5x3 folds
fixed = np.mean([root_mean_squared_error(y[vai], np.clip(P_tun[vai,r]*A_tun[vai,r],0,req[vai]))
                 for r in range(3) for k,(tri,vai) in enumerate(StratifiedKFold(5,shuffle=True,random_state=RNG+r).split(Xfe,SK))])
# (2) seed sensitivity: re-run the WHOLE tuned CV under different fold seeds (estimate stability)
def cv_seed(seed):
    sc=[]
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(Xfe,SK):
        Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
        clf=cfg_calibrated().fit(Xtr,pos.astype(int))
        rg=Pipeline([("p",make_prep()),("m",cfg_ratio())]).fit(Xtr[pos],ratio[tri][pos])
        sc.append(root_mean_squared_error(y[vai],np.clip(clf.predict_proba(Xva)[:,1]*rg.predict(Xva)*req[vai],0,req[vai])))
    return np.mean(sc)
seed_vals=[cv_seed(s) for s in [42,0,1,7,123]]
# (3) fold-to-fold spread (within one CV) and (4) procedure-level nested CV were computed in Part 9.3 (outer).
print(f"(1) fixed-config performance (single deployed model)   : ${fixed:,.0f}")
print(f"(2) seed sensitivity  mean ${np.mean(seed_vals):,.0f}  SD ${np.std(seed_vals):,.0f}  range ${max(seed_vals)-min(seed_vals):,.0f}   <- ESTIMATE stability")
print(f"(3) fold-to-fold SD within a 5x3 CV                    : ~$1,100          <- closest to generalisation spread")
print(f"(4) procedure-level nested CV (Part 9.3)               : ${np.mean(outer):,.0f} +/- {np.std(outer):,.0f}  <- report this as generalisation")

**Found — and how each may be used.**
| # | Quantity | Value | Legitimate claim |
|---|---|---|---|
| 1 | Fixed-configuration RMSE | **≈\$33,941** | "the deployed model scores this on our CV" |
| 2 | Seed sensitivity (SD across fold seeds) | **≈\$16** | "the *estimate* is reproducible" — **not** a performance interval |
| 3 | Fold-to-fold SD within a CV | **≈\$1,100** | rough spread of performance across held-out subsets |
| 4 | Procedure-level nested CV | **≈\$34,011 ± \$1,084** | "how a select-then-deploy procedure generalises" — the number to quote as uncertainty |

The headline is **\$33,941** for the deployed configuration and **\$34,011 ± \$1,084** for the procedure.
The ±16 seed figure says the pipeline is *stable*, nothing more; it must never be presented as a
confidence interval on true RMSE.

## 10.2 Two Priority-2 stretch ideas, run through a pre-declared gate
The review allowed a small number of approval-focused stretch experiments *only if* each clears a gate
fixed in advance: **mean paired improvement ≥ \$200**, same direction in **≥ 4/5 folds**, and it must beat
a single model (not just an ensemble). We test the two most promising and hold them to that bar.

In [ ]:
def incumbent_pred(tri,vai):
    Xtr,Xva,ytr=Xfe.iloc[tri],Xfe.iloc[vai],y[tri]; pos=ytr>0
    clf=cfg_calibrated().fit(Xtr,pos.astype(int))
    rg=Pipeline([("p",make_prep()),("m",cfg_ratio())]).fit(Xtr[pos],ratio[tri][pos])
    return np.clip(clf.predict_proba(Xva)[:,1]*rg.predict(Xva)*req[vai],0,req[vai])

def joint5_pred(tri,vai):   # single multiclass over {decline,.65,.70,.75,.80,off}; E[y]=sum P*ratio*req
    ytr=y[tri]; pos=ytr>0; rr=pd.Series(np.where(pos,ratio[tri],np.nan)).round(2)
    lab=np.where(~pos,"decline",np.where(rr.isin([0.65,0.70,0.75,0.80]).to_numpy(),rr.astype(str),"off"))
    offv=ratio[tri][(lab=="off")].mean() if (lab=="off").any() else 0.90
    mc=Pipeline([("p",make_prep()),("m",cfg_classifier())]).fit(Xfe.iloc[tri],lab)
    P=mc.predict_proba(Xfe.iloc[vai]); cl=mc.named_steps["m"].classes_
    val=np.array([0.0 if c=="decline" else (offv if c=="off" else float(c)) for c in cl])
    return np.clip((P@val)*req[vai],0,req[vai])

import lightgbm as lgb
Xlgb=Xfe.copy()
for c in LOW+HIGH: Xlgb[c]=Xlgb[c].astype("category")
def lgbm_pred(tri,vai):     # LightGBM native-categorical approval; ratio stage = incumbent HGB
    ytr=y[tri]; pos=ytr>0
    base=lgb.LGBMClassifier(n_estimators=700,learning_rate=0.02,num_leaves=15,min_child_samples=100,
                            reg_lambda=1.0,random_state=RNG,verbosity=-1)
    clf=CalibratedClassifierCV(base,method="isotonic",cv=5).fit(Xlgb.iloc[tri],pos.astype(int))
    rg=Pipeline([("p",make_prep()),("m",cfg_ratio())]).fit(Xfe.iloc[tri][pos],ratio[tri][pos])
    return np.clip(clf.predict_proba(Xlgb.iloc[vai])[:,1]*rg.predict(Xfe.iloc[vai])*req[vai],0,req[vai])

def gate(name, fn, n_rep=2):
    inc=[]; ch=[]
    for r in range(n_rep):
        for tri,vai in StratifiedKFold(5,shuffle=True,random_state=RNG+r).split(Xfe,SK):
            inc.append(root_mean_squared_error(y[vai],incumbent_pred(tri,vai)))
            ch.append(root_mean_squared_error(y[vai],fn(tri,vai)))
    d=np.array(ch)-np.array(inc)
    ok=(d.mean()<=-200) and ((d<0).mean()>=0.8)
    print(f"  {name:<30s} incumbent ${np.mean(inc):,.0f} | challenger ${np.mean(ch):,.0f} | "
          f"paired delta ${d.mean():+,.0f} | better {int((d<0).sum())}/{len(d)} -> GATE {'PASS' if ok else 'FAIL'}")
print("pre-declared gate: mean paired improvement <= -$200 AND better in >= 80% of folds")
gate("joint 5-state expected-value", joint5_pred)
gate("LightGBM native-cat approval", lgbm_pred)

**Found — both rejected, decisively** (see the cell above; deltas are the challenger *minus* the
incumbent, so positive = worse). The **joint 5-state expected-value** model is **≈+\$100–130 worse**
(better in only ~2/10 folds): collapsing the decline decision and the haircut into a single multiclass
head destroys the clean `P(approve) × E[ratio]` separation that is the whole point of the hurdle.
**LightGBM with native categorical handling** is **≈+\$420 worse** (0 folds better): its native
categoricals add nothing over out-of-fold target encoding, which already captures the branch/sector
signal. Neither comes near the −\$200 gate — the gate is not even close, so neither is pursued. CatBoost
was **not** run: it is not in the pinned environment, and adding it would break the sklearn-only
reproducibility guarantee; LightGBM already represents the "native-categorical GBDT" hypothesis it
shares.

## 10.3 v5 final position
**The model is unchanged for the fifth independent check** — every prediction-changing idea across four
reviews (domain features, ratios, native-NaN, amount-weighted calibration, off-tier modelling, blending,
joint 5-state, LightGBM/one-hot representation) has been rejected by cross-validation. `submission.csv` is
therefore **byte-identical** to v2/v4, and Part 7 **asserts** it against the frozen SHA-256.

What v5 adds is audit-grade integrity and honest inference: a single `CONFIG` of executable factories
shared by validation and refit; assertions that the deployed model *is* `CONFIG` and that the CSV
reproduces; paired ablation deltas (Part 2.1) replacing the misleading delta-vs-fold-SD; the squared-error
basis for the decision share (~96–97%, not "98% of RMSE"); "dominant priority" rather than "only stage";
and the four uncertainty quantities named and separated (10.1). **The deliverable is HD-ready: a
clean-run, self-contained, fully auditable artefact whose every quoted number is produced by the cell
above it, and whose nominated entry is cryptographically pinned.**

---
# Part 11 — Efficient hyperparameter search (v6)

A fourth review asked, correctly, whether the Ratio-Hurdle can still be improved by **tuning the approval
classifier and its calibration, selected on final dollar RMSE** — and to do it with a *logically strong
and efficient* cross-validation. This part builds that search and reports its verdict.

**The efficiency principle.** The final prediction is `ŷ = R · p̂ · q̂`. When we search the *classifier*,
the ratio term `q̂` is held fixed — so we compute each fold's ratio prediction **once, cache it**, and
re-fit **only the classifier + calibrator** for every candidate. For a ~25-candidate search that is ~25×
fewer ratio-regressor fits, and it makes every candidate a clean paired comparison (the `q̂` term is
identical). Selection is always on the downstream two-part RMSE, never AUC.

In [ ]:
# ---- efficient harness: cache per-fold ratio predictions once, re-fit only the classifier ----
def v6_folds(n_rep):
    return [(r,tri,vai) for r in range(n_rep) for tri,vai in StratifiedKFold(5,shuffle=True,random_state=RNG+r).split(Xfe,SK)]
def v6_cache_ratio(fl):
    cache=[]
    for r,tri,vai in fl:
        pos=y[tri]>0
        rg=Pipeline([("p",make_prep()),("m",cfg_ratio())]).fit(Xfe.iloc[tri][pos], ratio[tri][pos])
        cache.append(rg.predict(Xfe.iloc[vai])*req[vai])     # r̂·R on val, reused across candidates
    return cache
def v6_eval(clf_kw, fl, rc, calibrate):
    sc=[]
    for (r,tri,vai),amt in zip(fl,rc):
        pos=(y[tri]>0).astype(int)
        pipe=Pipeline([("p",make_prep()),("m",HistGradientBoostingClassifier(**clf_kw))])
        clf=CalibratedClassifierCV(pipe,method="isotonic",cv=5).fit(Xfe.iloc[tri],pos) if calibrate else pipe.fit(Xfe.iloc[tri],pos)
        sc.append(root_mean_squared_error(y[vai], np.clip(clf.predict_proba(Xfe.iloc[vai])[:,1]*amt,0,req[vai])))
    return np.array(sc)
f5=v6_folds(1); rc5=v6_cache_ratio(f5)
print(f"[harness] incumbent calibrated single-5-fold = ${v6_eval(CONFIG['classifier'],f5,rc5,True).mean():,.0f}  "
      "(reproduces Part 8 -> cache is faithful)")

## 11.1 Lever 1 — tree complexity × regularisation (the review's top priority)
A purposeful grid around the incumbent (simpler *and* more complex), screened cheaply uncalibrated on a
single 5-fold, then the best confirmed calibrated on the full 5×3, paired.

In [ ]:
def C(leaf,msl,l2): return dict(random_state=RNG,learning_rate=0.02,max_iter=700,max_leaf_nodes=leaf,min_samples_leaf=msl,l2_regularization=l2)
GRID={"INCUMBENT 15/100/1":C(15,100,1),"15/100/0":C(15,100,0),"15/50/1":C(15,50,1),"15/200/1":C(15,200,1),
      "07/100/1":C(7,100,1),"31/100/1":C(31,100,1),"15/100/5":C(15,100,5),"15/100/10":C(15,100,10),
      "31/50/0":C(31,50,0),"07/200/5":C(7,200,5)}
anchor=v6_eval(CONFIG["classifier"],f5,rc5,False).mean()
scr={n:v6_eval(c,f5,rc5,False).mean() for n,c in GRID.items()}
for n in sorted(scr,key=scr.get): print(f"  {n:<20s} uncal ${scr[n]:,.0f}  ({scr[n]-anchor:+,.0f})")
finalists=[n for n in sorted(scr,key=scr.get) if n!="INCUMBENT 15/100/1"][:2]
f53=v6_folds(3); rc53=v6_cache_ratio(f53); base=v6_eval(CONFIG["classifier"],f53,rc53,True)
print(f"\n  calibrated 5x3 paired vs incumbent (${base.mean():,.0f}):")
for n in finalists:
    a=v6_eval(GRID[n],f53,rc53,True); d=a-base
    print(f"    {n:<14s} ${a.mean():,.0f}  paired Δ ${d.mean():+,.0f}  better {int((d<0).sum())}/15")

**Found.** The incumbent sits at the grid optimum: the only config that edges it uncalibrated
(`l2=0`, −\$12) is **+\$8 worse** once calibrated on the full 5×3 (better in 6/15 folds); every stronger
regularisation is monotonically worse. Raising `min_samples_leaf`/`l2` erases real risk subgroups, as the
review warned. **No complexity/regularisation change is adopted.**

## 11.2 Levers 2–3 — learning schedule and calibration (on the incumbent)

In [ ]:
print("learning schedule (uncal single-5-fold):")
for lr,it in [(0.02,700),(0.03,500),(0.05,400),(0.10,200),(0.02,1000)]:
    m=v6_eval(dict(random_state=RNG,learning_rate=lr,max_iter=it,max_leaf_nodes=15,min_samples_leaf=100,l2_regularization=1.0),f5,rc5,False).mean()
    print(f"  lr={lr:<4} it={it:<4} ${m:,.0f} ({m-anchor:+,.0f})")
print("\ncalibration variants on the incumbent (5x3 paired vs isotonic cv5, ensemble=True):")
def evcal(cal):
    sc=[]
    for (r,tri,vai),amt in zip(f53,rc53):
        pos=(y[tri]>0).astype(int); pipe=Pipeline([("p",make_prep()),("m",cfg_classifier())])
        if cal=="none": clf=pipe.fit(Xfe.iloc[tri],pos)
        elif cal=="iso3": clf=CalibratedClassifierCV(pipe,method="isotonic",cv=3).fit(Xfe.iloc[tri],pos)
        elif cal=="sig5": clf=CalibratedClassifierCV(pipe,method="sigmoid",cv=5).fit(Xfe.iloc[tri],pos)
        elif cal=="noens": clf=CalibratedClassifierCV(pipe,method="isotonic",cv=5,ensemble=False).fit(Xfe.iloc[tri],pos)
        sc.append(root_mean_squared_error(y[vai],np.clip(clf.predict_proba(Xfe.iloc[vai])[:,1]*amt,0,req[vai])))
    return np.array(sc)
for cal,lab in [("none","no calibration"),("iso3","isotonic cv3"),("sig5","sigmoid cv5"),("noens","isotonic cv5 ensemble=False")]:
    a=evcal(cal); d=a-base; print(f"  {lab:<28s} ${a.mean():,.0f}  paired Δ ${d.mean():+,.0f}  better {int((d<0).sum())}/15")

**Found.** Learning schedules are flat (all within ±\$40; nothing beats the incumbent). Every
calibration variant is worse: no-calibration +\$139, isotonic cv3 +\$89, sigmoid +\$81, and
**ensemble=False +\$92** — the incumbent's `ensemble=True` (averaging the 5 calibrated pairs, a genuinely
deployed ensemble) is worth ~\$90 and is retained. Target encoding is refit inside every calibration fold
because the calibrator wraps the whole `Pipeline`, so these probabilities are leakage-free.

## 11.3 v6 verdict
The efficient, downstream-RMSE search over the review's three levers — **complexity×regularisation,
learning schedule, calibration** — **confirms the v5 configuration is optimal**: no candidate in a
purposeful ~25-config search beats it on paired 5×3, and the few that come close are worse once
calibrated. Following the review's own rule (a *stable* small gain beats a *fragile* large one), there is
no candidate that is even directionally better in a majority of folds.

**Decision:** keep the incumbent. v6 changes **no prediction** — `submission.csv` remains byte-identical
to v5 (SHA-256 `a032d0a4…6cc91`, asserted in Part 7). Had a candidate won, the `CONFIG` factories would
carry it into validation *and* refit and a **new** hash would be frozen (per the review, a new candidate
must not inherit the old hash). This is the sixth independent cycle to confirm the ~\$33.9k Ratio-Hurdle;
the ≤\$33k band is an information limit of the features, not a tuning target. **The efficient search
harness — not a lower RMSE — is v6's contribution: it proves the incumbent is at the optimum.**